# Tail-Aware Soil Organic Content Regression

This notebook builds an end-to-end, leakage-safe machine learning pipeline for predicting `property_organic_content` with RMSE as the evaluation metric.

The design is competition-oriented:

- Strong tabular regressors: CatBoost, LightGBM, XGBoost.
- Fold-safe preprocessing, target encoding, imputation, model fitting, and validation.
- Explicit diagnostics for very low and very high organic-content tails.
- Conservative residual correction, bin-wise calibration, weighted training experiments, and ensemble selection.
- Final `submission.csv` plus useful artifacts for analysis.

No external data, no AutoML, and no automated feature engineering libraries are used.

## 1. Imports

The imports are intentionally standard for Kaggle tabular regression work. Optional libraries such as Optuna are imported only if available.

In [ ]:
# ============================================================
# Setup Cell: Install Missing Libraries + Imports + Config
# ============================================================

import sys
import subprocess
import importlib.util

def install_if_missing(module_name, package_name=None):
    """
    Install package only if the module is missing.
    Useful for Kaggle/Colab notebooks where some libraries may not exist.
    """
    if package_name is None:
        package_name = module_name

    if importlib.util.find_spec(module_name) is None:
        print(f"{module_name} is not available. Installing {package_name}...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", package_name
        ])
        print(f"{package_name} installed successfully.")
    else:
        print(f"{module_name} is already available.")


# Install required optional ML libraries
install_if_missing("catboost", "catboost")
install_if_missing("optuna", "optuna")


# ============================================================
# Standard Imports
# ============================================================

import os
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge, ElasticNet, LinearRegression
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance


# ============================================================
# ML Library Imports
# ============================================================

try:
    from catboost import CatBoostRegressor, Pool
    CATBOOST_AVAILABLE = True
except Exception as e:
    CATBOOST_AVAILABLE = False
    print(f"CatBoost is not available: {e}")

try:
    from lightgbm import LGBMRegressor
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except Exception as e:
    LGBM_AVAILABLE = False
    print(f"LightGBM is not available: {e}")

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception as e:
    XGB_AVAILABLE = False
    print(f"XGBoost is not available: {e}")

try:
    import optuna
    OPTUNA_AVAILABLE = True
except Exception as e:
    OPTUNA_AVAILABLE = False
    print(f"Optuna is not available: {e}")


# ============================================================
# Global Config
# ============================================================

warnings.filterwarnings("ignore")

SEED = 42
N_SPLITS = 3
TARGET = "property_organic_content"
ID_COL = "sample_id"
EPS = 1e-6

# Runtime switches
RUN_GROUP_DIAGNOSTIC = True
RUN_KMEANS_FEATURES = True
RUN_PCA_META_FEATURES = True
RUN_LOG_TARGET_EXPERIMENT = True
RUN_WEIGHTED_EXPERIMENT = True

# Since Optuna will be installed above, we can enable it
RUN_OPTUNA = True
N_OPTUNA_TRIALS = 20


# ============================================================
# Reproducibility
# ============================================================

np.random.seed(SEED)
random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)


# ============================================================
# Display Settings
# ============================================================

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


# ============================================================
# Final Check
# ============================================================

if RUN_OPTUNA and not OPTUNA_AVAILABLE:
    print("RUN_OPTUNA=True but Optuna is not available. Setting RUN_OPTUNA=False.")
    RUN_OPTUNA = False

print("Setup complete")
print(f"CatBoost: {CATBOOST_AVAILABLE} | LightGBM: {LGBM_AVAILABLE} | XGBoost: {XGB_AVAILABLE} | Optuna: {OPTUNA_AVAILABLE}")

## 2. Load Data

The competition files are loaded from the working directory. On Kaggle, this usually means `/kaggle/input/<competition-name>/`; locally it may be the notebook directory.

In [ ]:
# ============================================================
# Robust Data Loader for Google Colab / Kaggle / Local
# ============================================================

from pathlib import Path
import os
import zipfile
import pandas as pd

REQUIRED_FILES = [
    "train.csv",
    "test.csv",
    "sample_submission.csv",
]

# Optional Google Drive setup
# Set USE_GOOGLE_DRIVE = True kalau file kamu ada di Google Drive
USE_GOOGLE_DRIVE = False

# Kalau pakai Google Drive, isi folder spesifiknya supaya tidak scan seluruh Drive
# Contoh:
# DRIVE_DATA_DIR = Path("/content/drive/MyDrive/soil_competition")
DRIVE_DATA_DIR = None


def is_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False


def mount_drive_if_needed():
    if USE_GOOGLE_DRIVE and is_colab():
        from google.colab import drive
        drive.mount("/content/drive")


def extract_zip_files(search_dirs):
    """
    Extract zip files found in common runtime folders.
    Useful kalau kamu upload dataset dalam bentuk .zip.
    """
    extract_dirs = []

    for base in search_dirs:
        if not base.exists() or not base.is_dir():
            continue

        for zip_path in base.glob("*.zip"):
            extract_dir = base / f"extracted_{zip_path.stem}"
            extract_dir.mkdir(parents=True, exist_ok=True)

            print(f"Extracting zip: {zip_path} -> {extract_dir}")
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(extract_dir)

            extract_dirs.append(extract_dir)

    return extract_dirs


def build_candidate_paths():
    candidate_paths = []

    common_dirs = [
        Path("."),
        Path("/content"),
        Path("/content/data"),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    for path in common_dirs:
        if path.exists():
            candidate_paths.append(path)

    if DRIVE_DATA_DIR is not None and Path(DRIVE_DATA_DIR).exists():
        candidate_paths.append(Path(DRIVE_DATA_DIR))

    # Add one-level subfolders
    expanded_paths = []

    for base in candidate_paths:
        expanded_paths.append(base)

        if base.exists() and base.is_dir():
            try:
                expanded_paths.extend([p for p in base.iterdir() if p.is_dir()])
            except Exception:
                pass

    # Extract zip files from likely upload folders
    zip_extract_dirs = extract_zip_files([
        Path("."),
        Path("/content"),
        Path("/content/data"),
        Path("/kaggle/working"),
    ])

    expanded_paths.extend(zip_extract_dirs)

    # Add one-level subfolders from extracted dirs
    for base in zip_extract_dirs:
        if base.exists() and base.is_dir():
            expanded_paths.extend([p for p in base.iterdir() if p.is_dir()])

    # Remove duplicates while preserving order
    unique_paths = []
    seen = set()

    for path in expanded_paths:
        resolved = str(path.resolve()) if path.exists() else str(path)
        if resolved not in seen:
            unique_paths.append(path)
            seen.add(resolved)

    return unique_paths


def find_file(filename, candidates):
    """
    Search direct path, one-level folders, then recursive search in safe folders.
    """
    checked_paths = []

    # Direct check
    for base in candidates:
        path = base / filename
        checked_paths.append(path)
        if path.exists():
            return path

    # Recursive search only in safe/small folders
    safe_recursive_roots = [
        Path("."),
        Path("/content"),
        Path("/content/data"),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    if DRIVE_DATA_DIR is not None:
        safe_recursive_roots.append(Path(DRIVE_DATA_DIR))

    for root in safe_recursive_roots:
        if not root.exists() or not root.is_dir():
            continue

        try:
            matches = list(root.rglob(filename))
            if len(matches) > 0:
                return matches[0]
        except Exception:
            pass

    return None


def print_runtime_files():
    """
    Diagnostic helper supaya kelihatan file apa aja yang ada.
    """
    print("\nCurrent working directory:", Path(".").resolve())

    folders_to_check = [
        Path("."),
        Path("/content"),
        Path("/content/data"),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    if DRIVE_DATA_DIR is not None:
        folders_to_check.append(Path(DRIVE_DATA_DIR))

    print("\nVisible files/folders:")

    for folder in folders_to_check:
        if folder.exists():
            print(f"\n{folder}:")
            try:
                items = list(folder.iterdir())[:30]
                for item in items:
                    print(" -", item.name)
            except Exception as e:
                print(" Could not list:", e)


# ============================================================
# Main loading logic
# ============================================================

mount_drive_if_needed()

candidate_paths = build_candidate_paths()

found_paths = {}
missing_files = []

for filename in REQUIRED_FILES:
    path = find_file(filename, candidate_paths)

    if path is None:
        missing_files.append(filename)
    else:
        found_paths[filename] = path


# If running in Colab and files are missing, open upload prompt
if missing_files and is_colab():
    print("Missing files:", missing_files)
    print("Please upload these files now:", REQUIRED_FILES)

    from google.colab import files
    uploaded = files.upload()

    # Extract uploaded zip files if any
    for uploaded_name in uploaded.keys():
        uploaded_path = Path(uploaded_name)

        if uploaded_path.suffix.lower() == ".zip":
            extract_dir = Path("/content") / f"extracted_{uploaded_path.stem}"
            extract_dir.mkdir(parents=True, exist_ok=True)

            print(f"Extracting uploaded zip: {uploaded_path} -> {extract_dir}")
            with zipfile.ZipFile(uploaded_path, "r") as z:
                z.extractall(extract_dir)

    # Rebuild candidates after upload/extract
    candidate_paths = build_candidate_paths()

    found_paths = {}
    missing_files = []

    for filename in REQUIRED_FILES:
        path = find_file(filename, candidate_paths)

        if path is None:
            missing_files.append(filename)
        else:
            found_paths[filename] = path


if missing_files:
    print_runtime_files()
    raise FileNotFoundError(
        "Still could not find these files: "
        + str(missing_files)
        + "\n\nFix options:\n"
        + "1. In Colab, upload train.csv, test.csv, sample_submission.csv directly.\n"
        + "2. Or upload a zip containing those three files.\n"
        + "3. Or mount Google Drive and set USE_GOOGLE_DRIVE=True plus DRIVE_DATA_DIR.\n"
        + "4. In Kaggle, add the competition dataset from Add Data."
    )


train_path = found_paths["train.csv"]
test_path = found_paths["test.csv"]
submission_path = found_paths["sample_submission.csv"]

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(submission_path)

print(f"train path: {train_path}")
print(f"test path: {test_path}")
print(f"sample_submission path: {submission_path}")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())
display(test.head())
display(sample_submission.head())

In [ ]:
# Basic schema checks.
assert TARGET in train.columns, f"{TARGET} must exist in train"
assert TARGET not in test.columns, f"{TARGET} should not exist in test"
assert ID_COL in train.columns and ID_COL in test.columns and ID_COL in sample_submission.columns

train_feature_cols = [c for c in train.columns if c != TARGET]
test_feature_cols = list(test.columns)

missing_in_test = sorted(set(train_feature_cols) - set(test_feature_cols))
extra_in_test = sorted(set(test_feature_cols) - set(train_feature_cols))
print("Feature columns missing in test:", missing_in_test)
print("Extra columns in test:", extra_in_test)

assert len(missing_in_test) == 0, "Test is missing train feature columns."

if len(extra_in_test) > 0:
    print("Warning: test contains extra columns. They will be ignored unless explicitly used.")

if sample_submission[ID_COL].equals(test[ID_COL]):
    print("sample_submission sample_id perfectly aligns with test sample_id.")
else:
    print("Warning: sample_submission sample_id order differs from test. Submission will be aligned by sample_id.")

X = train.drop([TARGET], axis=1).copy()
y = train[TARGET].copy()
test_features = test.copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("test_features shape:", test_features.shape)

## 3. Data Cleaning, Preprocessing, and EDA

The EDA below focuses on issues that matter for modeling: missingness, train/test distribution drift, categorical cardinality, target tails, spectral availability, and possible leakage traps.

In [ ]:
# Define feature groups from column names.
ALL_FEATURES = [c for c in train.columns if c != TARGET]

spectral_A_cols = sorted([c for c in ALL_FEATURES if c.startswith("spectral_band_A_PC_")], key=lambda x: int(x.split("_")[-1]))
spectral_B_cols = sorted([c for c in ALL_FEATURES if c.startswith("spectral_band_B_PC_")], key=lambda x: int(x.split("_")[-1]))
geo_cols = [c for c in ["latitude", "longitude"] if c in ALL_FEATURES]
soil_numeric_cols = [
    c for c in [
        "property_particle_coarse",
        "property_particle_fine",
        "property_acidity_index",
        "cation_Ca",
        "cation_Mg",
        "cation_Na",
        "cation_exchange_capacity",
    ] if c in ALL_FEATURES
]
flag_cols = [c for c in ["has_band_A_spectrum", "has_band_B_spectrum"] if c in ALL_FEATURES]

base_categorical_cols = [
    c for c in [
        "source_id",
        "sampling_strategy",
        "geo_zone_macro",
        "geo_zone_meso",
        "geo_zone_micro",
        "land_cover_type",
        "biome",
        "parent_rock_type",
        "sampling_depth_cm",
        "has_band_A_spectrum",
        "has_band_B_spectrum",
    ] if c in ALL_FEATURES
]

base_numeric_cols = [c for c in ALL_FEATURES if c not in base_categorical_cols + [ID_COL]]

print("Categorical columns:", base_categorical_cols)
print("Numeric columns:", base_numeric_cols)
print("Soil numeric columns:", soil_numeric_cols)
print("Geo columns:", geo_cols)
print("Spectral A columns:", len(spectral_A_cols), spectral_A_cols[:3], "...")
print("Spectral B columns:", len(spectral_B_cols), spectral_B_cols[:3], "...")

In [ ]:
def missing_summary(df, name):
    miss = df.isna().sum()
    pct = miss / len(df) * 100
    out = pd.DataFrame({
        "dataset": name,
        "missing_count": miss,
        "missing_pct": pct,
        "dtype": df.dtypes.astype(str),
        "nunique": df.nunique(dropna=False),
    }).sort_values("missing_pct", ascending=False)
    return out

train_miss = missing_summary(train, "train")
test_miss = missing_summary(test, "test")
miss_compare = train_miss[["missing_count", "missing_pct"]].rename(columns=lambda c: f"train_{c}").join(
    test_miss[["missing_count", "missing_pct"]].rename(columns=lambda c: f"test_{c}"), how="outer"
).fillna(0)
miss_compare["pct_diff_train_minus_test"] = miss_compare["train_missing_pct"] - miss_compare["test_missing_pct"]

print("Top missing columns in train:")
display(train_miss.head(25))
print("Train/test missingness comparison:")
display(miss_compare.sort_values("train_missing_pct", ascending=False).head(35))

print("Duplicated full rows in train:", train.duplicated().sum())
print("Duplicated full rows in test:", test.duplicated().sum())
print("Duplicated sample_id in train:", train[ID_COL].duplicated().sum())
print("Duplicated sample_id in test:", test[ID_COL].duplicated().sum())

print("Data types:")
display(pd.DataFrame({"dtype": train.dtypes.astype(str), "nunique_train": train.nunique(dropna=False)}))

### Missingness interpretation

Missingness is not just a nuisance here. It likely carries information about lab availability, source protocols, geography, spectral equipment coverage, and sampling campaigns. Therefore, the modeling pipeline keeps missingness indicators instead of forcing every missing value into a single median-smoothed fog bank.

In [ ]:
plt.figure(figsize=(12, 7))
miss_plot = miss_compare.sort_values("train_missing_pct", ascending=False).head(35)
sns.barplot(x=miss_plot["train_missing_pct"], y=miss_plot.index)
plt.title("Top missing-value percentages in train")
plt.xlabel("Missing %")
plt.ylabel("Feature")
plt.show()

# Compact missingness heatmap sample.
sample_for_heatmap = train.sample(min(len(train), 800), random_state=SEED)
plt.figure(figsize=(14, 6))
sns.heatmap(sample_for_heatmap.isna(), cbar=False)
plt.title("Missingness heatmap sample from train")
plt.xlabel("Columns")
plt.ylabel("Sampled rows")
plt.show()

In [ ]:
# Categorical cardinality and train/test level comparison.
cat_cardinality = []
cat_level_diffs = {}

for col in base_categorical_cols:
    train_levels = set(train[col].fillna("Missing").astype(str).unique())
    test_levels = set(test[col].fillna("Missing").astype(str).unique())
    unseen_in_test = sorted(test_levels - train_levels)
    missing_from_test = sorted(train_levels - test_levels)
    cat_cardinality.append({
        "column": col,
        "train_nunique": len(train_levels),
        "test_nunique": len(test_levels),
        "unseen_test_levels_count": len(unseen_in_test),
        "train_only_levels_count": len(missing_from_test),
        "unseen_test_levels_examples": unseen_in_test[:10],
    })
    cat_level_diffs[col] = {
        "unseen_in_test": unseen_in_test,
        "train_only": missing_from_test,
    }

cat_cardinality = pd.DataFrame(cat_cardinality).sort_values("train_nunique", ascending=False)
display(cat_cardinality)

print("Strategy: categorical NaN -> 'Missing'; literal 'Unknown' remains a valid category because it may encode survey/source information.")
print("Unseen categories are handled by CatBoost natively and by OrdinalEncoder unknown_value=-1 for LGBM/XGB.")

In [ ]:
# Numeric distribution summary.
numeric_summary = train[base_numeric_cols + [TARGET]].describe(percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]).T
numeric_summary["skew"] = train[base_numeric_cols + [TARGET]].skew(numeric_only=True)
numeric_summary["missing_pct"] = train[base_numeric_cols + [TARGET]].isna().mean() * 100
display(numeric_summary)

# Target quantile table.
target_quantiles = y.quantile([0, .01, .05, .10, .25, .50, .75, .90, .95, .99, 1.0]).to_frame("target_quantile")
display(target_quantiles)

low_5 = y.quantile(0.05)
high_95 = y.quantile(0.95)
low_10 = y.quantile(0.10)
high_90 = y.quantile(0.90)
print(f"Very low zone <= 5th pct: {low_5:.5f}")
print(f"Low zone <= 10th pct: {low_10:.5f}")
print(f"High zone >= 90th pct: {high_90:.5f}")
print(f"Very high zone >= 95th pct: {high_95:.5f}")
print(f"Target skewness: {stats.skew(y.dropna()):.4f}")

### Target distribution and tails

The objective is RMSE, so large misses on rare high-organic samples can dominate the score. We will not remove outliers. Instead, we diagnose the tails and later test tail-aware corrections.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(y, bins=60, kde=True, ax=axes[0])
axes[0].set_title("Target distribution")
axes[0].set_xlabel(TARGET)

sns.histplot(np.log1p(y.clip(lower=0)), bins=60, kde=True, ax=axes[1])
axes[1].set_title("log1p(target) distribution")
axes[1].set_xlabel(f"log1p({TARGET})")

sns.boxplot(x=y, ax=axes[2])
axes[2].set_title("Target boxplot")
axes[2].set_xlabel(TARGET)
plt.tight_layout()
plt.show()

In [ ]:
def plot_target_by_category(df, col, target=TARGET, top_n=20):
    order = df.groupby(col)[target].median().sort_values(ascending=False).index[:top_n]
    plt.figure(figsize=(12, max(4, 0.35 * len(order))))
    sns.boxplot(data=df[df[col].isin(order)], y=col, x=target, order=order)
    plt.title(f"{target} by {col} | top {top_n} categories by median target")
    plt.show()

for col in ["source_id", "biome", "land_cover_type", "parent_rock_type", "geo_zone_macro"]:
    if col in train.columns:
        plot_target_by_category(train.assign(**{col: train[col].fillna("Missing").astype(str)}), col, top_n=20)

In [ ]:
# Correlation heatmap for numerical features.
corr_cols = [c for c in base_numeric_cols + [TARGET] if train[c].notna().sum() > 10]
corr = train[corr_cols].corr(method="spearman")
plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, square=False)
plt.title("Spearman correlation heatmap: numeric features + target")
plt.show()

# Spectral-target correlation exploration.
spectral_corr = train[spectral_A_cols + spectral_B_cols + [TARGET]].corr(method="spearman")[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
display(spectral_corr.to_frame("spearman_corr_with_target"))

plt.figure(figsize=(10, 5))
spectral_corr.sort_values().plot(kind="bar")
plt.title("Spectral PC Spearman correlation with target")
plt.ylabel("Correlation")
plt.tight_layout()
plt.show()

In [ ]:
# Train/test distribution comparison for important numeric fields.
important_compare_cols = soil_numeric_cols + geo_cols + spectral_A_cols[:3] + spectral_B_cols[:3]
for col in important_compare_cols:
    if col not in train.columns or col not in test.columns:
        continue
    plt.figure(figsize=(9, 4))
    sns.kdeplot(train[col].dropna(), label="train", fill=False)
    sns.kdeplot(test[col].dropna(), label="test", fill=False)
    plt.title(f"Train/test distribution comparison: {col}")
    plt.legend()
    plt.show()

## 4. Feature Engineering and Dimensionality Reduction

Feature engineering is intentionally conservative and fold-safe. Operations that use the target, such as target encoding, are done inside the cross-validation scheme. Operations that do not use the target, such as category normalization and frequency encoding, can be fit on train+test for consistency.

In [ ]:
def normalize_categoricals(df, categorical_cols):
    """Normalize categorical columns without turning literal 'Unknown' into NaN."""
    out = df.copy()
    for col in categorical_cols:
        if col in out.columns:
            out[col] = out[col].astype("object")
            out[col] = out[col].where(out[col].notna(), "Missing")
            out[col] = out[col].astype(str).str.strip()
            out[col] = out[col].replace({"": "Missing", "nan": "Missing", "None": "Missing"})
    return out


def safe_div(a, b, eps=EPS):
    return a / (b + eps)


def parse_depth_series(s):
    """Parse strings such as '0-20', '20', '0 – 30 cm' into depth features."""
    s = s.fillna("Missing").astype(str).str.lower().str.replace("cm", "", regex=False).str.strip()
    # Extract first two numeric values when available.
    extracted = s.str.extract(r"(?P<min>-?\d+\.?\d*)\s*(?:-|–|to)?\s*(?P<max>-?\d+\.?\d*)?")
    depth_min = pd.to_numeric(extracted["min"], errors="coerce")
    depth_max = pd.to_numeric(extracted["max"], errors="coerce")
    depth_max = depth_max.fillna(depth_min)
    depth_mid = (depth_min + depth_max) / 2
    depth_range = depth_max - depth_min
    return depth_min, depth_max, depth_mid, depth_range


def add_row_missing_features(df, feature_cols):
    out = df.copy()
    non_id_cols = [c for c in feature_cols if c in out.columns and c != ID_COL]
    out["total_missing_count"] = out[non_id_cols].isna().sum(axis=1)
    out["missing_ratio"] = out["total_missing_count"] / max(len(non_id_cols), 1)
    return out


def feature_engineering(df, is_train=False):
    """Create target-independent features. Safe to apply consistently to train and test."""
    out = df.copy()
    out = normalize_categoricals(out, base_categorical_cols)
    original_feature_cols = [c for c in out.columns if c != TARGET]
    out = add_row_missing_features(out, original_feature_cols)

    # Important missing indicators.
    important_missing_cols = soil_numeric_cols + geo_cols
    for col in important_missing_cols:
        if col in out.columns:
            out[f"{col}_is_missing"] = out[col].isna().astype(np.int8)

    # Spectral missingness and availability.
    if spectral_A_cols:
        out["spectral_A_missing_count"] = out[spectral_A_cols].isna().sum(axis=1)
        out["has_any_band_A_value"] = out[spectral_A_cols].notna().any(axis=1).astype(np.int8)
        out["has_all_band_A_missing"] = out[spectral_A_cols].isna().all(axis=1).astype(np.int8)
    if spectral_B_cols:
        out["spectral_B_missing_count"] = out[spectral_B_cols].isna().sum(axis=1)
        out["has_any_band_B_value"] = out[spectral_B_cols].notna().any(axis=1).astype(np.int8)
        out["has_all_band_B_missing"] = out[spectral_B_cols].isna().all(axis=1).astype(np.int8)

    if set(["latitude", "longitude"]).issubset(out.columns):
        out["has_geo_coordinates"] = out[["latitude", "longitude"]].notna().all(axis=1).astype(np.int8)
        out["lat_missing"] = out["latitude"].isna().astype(np.int8)
        out["lon_missing"] = out["longitude"].isna().astype(np.int8)
        out["latitude_round_1"] = out["latitude"].round(1)
        out["longitude_round_1"] = out["longitude"].round(1)
        out["latitude_round_2"] = out["latitude"].round(2)
        out["longitude_round_2"] = out["longitude"].round(2)
        out["lat_lon_interaction"] = out["latitude"] * out["longitude"]
        out["abs_latitude"] = out["latitude"].abs()
        out["is_southern_hemisphere"] = (out["latitude"] < 0).astype(np.int8)
        out["geo_bin_1"] = out["latitude_round_1"].astype(str) + "_" + out["longitude_round_1"].astype(str)

    # Spectral row-wise meta-features.
    for prefix, cols in [("A", spectral_A_cols), ("B", spectral_B_cols)]:
        if not cols:
            continue
        vals = out[cols]
        out[f"spectral_{prefix}_mean"] = vals.mean(axis=1, skipna=True)
        out[f"spectral_{prefix}_std"] = vals.std(axis=1, skipna=True)
        out[f"spectral_{prefix}_min"] = vals.min(axis=1, skipna=True)
        out[f"spectral_{prefix}_max"] = vals.max(axis=1, skipna=True)
        out[f"spectral_{prefix}_sum"] = vals.sum(axis=1, skipna=True)
        out[f"spectral_{prefix}_abs_sum"] = vals.abs().sum(axis=1, skipna=True)
        out[f"spectral_{prefix}_l2_norm"] = np.sqrt((vals.fillna(0) ** 2).sum(axis=1))
        # Stable ratios/interactions among early PCs. Denominators receive EPS protection.
        if len(cols) >= 3:
            out[f"spectral_{prefix}_pc1_pc2_ratio"] = safe_div(out[cols[0]], out[cols[1]])
            out[f"spectral_{prefix}_pc1_pc3_ratio"] = safe_div(out[cols[0]], out[cols[2]])
            out[f"spectral_{prefix}_pc1_x_pc2"] = out[cols[0]] * out[cols[1]]
            out[f"spectral_{prefix}_pc2_x_pc3"] = out[cols[1]] * out[cols[2]]

    # Soil chemistry / physical features.
    coarse = out.get("property_particle_coarse")
    fine = out.get("property_particle_fine")
    acidity = out.get("property_acidity_index")
    ca = out.get("cation_Ca")
    mg = out.get("cation_Mg")
    na = out.get("cation_Na")
    cec = out.get("cation_exchange_capacity")

    if coarse is not None and fine is not None:
        out["coarse_to_fine_ratio"] = safe_div(coarse, fine)
        out["fine_to_coarse_ratio"] = safe_div(fine, coarse)
        out["total_particle"] = coarse + fine
        out["coarse_fraction"] = safe_div(coarse, out["total_particle"])
        out["fine_fraction"] = safe_div(fine, out["total_particle"])

    if ca is not None and mg is not None:
        out["ca_mg_ratio"] = safe_div(ca, mg)
    if ca is not None and na is not None:
        out["ca_na_ratio"] = safe_div(ca, na)
    if mg is not None and na is not None:
        out["mg_na_ratio"] = safe_div(mg, na)
    if ca is not None and mg is not None and na is not None:
        out["cation_sum"] = ca + mg + na
        if cec is not None:
            out["cec_per_cation_sum"] = safe_div(cec, out["cation_sum"])
        if acidity is not None:
            out["acidity_x_cation_sum"] = acidity * out["cation_sum"]
    if acidity is not None and cec is not None:
        out["acidity_x_cec"] = acidity * cec

    # Log transforms for positive-valued chemistry and texture features.
    log_candidates = soil_numeric_cols + ["total_particle", "cation_sum"]
    for col in log_candidates:
        if col in out.columns:
            out[f"log1p_{col}"] = np.log1p(out[col].clip(lower=0))

    # Depth parsing.
    if "sampling_depth_cm" in out.columns:
        dmin, dmax, dmid, drange = parse_depth_series(out["sampling_depth_cm"])
        out["depth_min"] = dmin
        out["depth_max"] = dmax
        out["depth_mid"] = dmid
        out["depth_range"] = drange

    # Geo/category combinations. These remain categorical.
    def combine(a, b, new_name):
        if a in out.columns and b in out.columns:
            out[new_name] = out[a].astype(str) + "__" + out[b].astype(str)

    combine("geo_zone_macro", "geo_zone_meso", "geo_macro_meso")
    combine("geo_zone_meso", "geo_zone_micro", "geo_meso_micro")
    combine("source_id", "geo_zone_macro", "source_geo_macro")
    combine("source_id", "biome", "source_biome")
    combine("biome", "land_cover_type", "biome_land_cover")
    combine("parent_rock_type", "biome", "parent_rock_biome")
    combine("sampling_strategy", "sampling_depth_cm", "strategy_depth")

    return out

train_fe = feature_engineering(train.drop(columns=[TARGET]), is_train=True)
test_fe = feature_engineering(test_features, is_train=False)

print("Feature engineered train shape:", train_fe.shape)
print("Feature engineered test shape:", test_fe.shape)
display(train_fe.head())

In [ ]:
# Frequency encoding using train+test feature distributions only. This is target-independent.
def add_frequency_encoding(train_df, test_df, cols):
    train_out = train_df.copy()
    test_out = test_df.copy()
    full = pd.concat([train_out[cols], test_out[cols]], axis=0, ignore_index=True)
    n_total = len(full)
    for col in cols:
        if col not in train_out.columns or col not in test_out.columns:
            continue
        freq = full[col].fillna("Missing").astype(str).value_counts(dropna=False) / n_total
        train_out[f"{col}_freq"] = train_out[col].fillna("Missing").astype(str).map(freq).fillna(0).astype(float)
        test_out[f"{col}_freq"] = test_out[col].fillna("Missing").astype(str).map(freq).fillna(0).astype(float)
    return train_out, test_out

freq_cols = [c for c in [
    "source_id",
    "geo_zone_micro",
    "geo_zone_meso",
    "biome",
    "parent_rock_type",
    "land_cover_type",
    "source_biome",
    "source_geo_macro",
    "geo_macro_meso",
    "geo_meso_micro",
] if c in train_fe.columns]

train_fe, test_fe = add_frequency_encoding(train_fe, test_fe, freq_cols)
print("Added frequency encodings for:", freq_cols)
print(train_fe.shape, test_fe.shape)

In [ ]:
# Optional PCA meta-components on spectral features. Since the raw spectral features are already PCs,
# these are added as meta-features instead of replacing the original PCs.
def add_pca_meta_features(train_df, test_df, cols, prefix, n_components=3):
    if not cols or len(cols) < n_components:
        return train_df, test_df, []
    train_out = train_df.copy()
    test_out = test_df.copy()

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    pca = PCA(n_components=n_components, random_state=SEED)

    full = pd.concat([train_out[cols], test_out[cols]], axis=0, ignore_index=True)
    full_imp = imputer.fit_transform(full)
    full_scaled = scaler.fit_transform(full_imp)
    full_pca = pca.fit_transform(full_scaled)

    new_cols = []
    for i in range(n_components):
        new_col = f"{prefix}_pca_meta_{i+1}"
        new_cols.append(new_col)
        train_out[new_col] = full_pca[:len(train_out), i]
        test_out[new_col] = full_pca[len(train_out):, i]

    print(f"{prefix}: added {n_components} PCA meta features. Explained variance:", np.round(pca.explained_variance_ratio_, 4))
    return train_out, test_out, new_cols

pca_meta_cols = []
if RUN_PCA_META_FEATURES:
    train_fe, test_fe, new_cols_A = add_pca_meta_features(train_fe, test_fe, spectral_A_cols, "spectral_A", n_components=min(5, len(spectral_A_cols)))
    train_fe, test_fe, new_cols_B = add_pca_meta_features(train_fe, test_fe, spectral_B_cols, "spectral_B", n_components=min(5, len(spectral_B_cols)))
    pca_meta_cols.extend(new_cols_A + new_cols_B)

print("PCA meta cols:", pca_meta_cols)

In [ ]:
# Unsupervised cluster features. This uses train+test covariates only, not target.
# It is a transductive, target-independent transformation often used in Kaggle notebooks.
# If your competition disallows any test-feature fitting, set RUN_KMEANS_FEATURES=False.
def add_kmeans_features(train_df, test_df, cols, prefix, ks=(4, 6, 8, 12)):
    if not cols:
        return train_df, test_df, []
    train_out = train_df.copy()
    test_out = test_df.copy()
    full = pd.concat([train_out[cols], test_out[cols]], axis=0, ignore_index=True)

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_full = scaler.fit_transform(imputer.fit_transform(full))

    added = []
    for k in ks:
        if k >= len(full):
            continue
        km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
        labels = km.fit_predict(X_full)
        dist = km.transform(X_full).min(axis=1)
        label_col = f"{prefix}_kmeans_{k}"
        dist_col = f"{prefix}_kmeans_{k}_min_dist"
        train_out[label_col] = labels[:len(train_out)].astype(str)
        test_out[label_col] = labels[len(train_out):].astype(str)
        train_out[dist_col] = dist[:len(train_out)]
        test_out[dist_col] = dist[len(train_out):]
        added.extend([label_col, dist_col])
    return train_out, test_out, added

cluster_cols_added = []
if RUN_KMEANS_FEATURES:
    soil_cluster_cols = [c for c in soil_numeric_cols + geo_cols if c in train_fe.columns]
    spectral_A_cluster_cols = [c for c in spectral_A_cols if c in train_fe.columns]
    spectral_B_cluster_cols = [c for c in spectral_B_cols if c in train_fe.columns]

    train_fe, test_fe, added = add_kmeans_features(train_fe, test_fe, soil_cluster_cols, "soil_geo", ks=(4, 6, 8, 12))
    cluster_cols_added.extend(added)
    train_fe, test_fe, added = add_kmeans_features(train_fe, test_fe, spectral_A_cluster_cols, "spectral_A", ks=(4, 6, 8, 12))
    cluster_cols_added.extend(added)
    train_fe, test_fe, added = add_kmeans_features(train_fe, test_fe, spectral_B_cluster_cols, "spectral_B", ks=(4, 6, 8))
    cluster_cols_added.extend(added)

print("Cluster features added:", cluster_cols_added)
print("Shapes after unsupervised features:", train_fe.shape, test_fe.shape)

### Cluster diagnostics

Clusters are not the final objective. They are used as compact regime indicators: source/geography/chemistry/spectral neighborhoods may capture organic-content regimes that a single global model smooths over.

In [ ]:
for col in [c for c in cluster_cols_added if c.endswith("kmeans_4")][:3]:
    tmp = pd.DataFrame({col: train_fe[col], TARGET: y})
    plt.figure(figsize=(9, 4))
    sns.boxplot(data=tmp, x=col, y=TARGET)
    plt.title(f"Target distribution by cluster: {col}")
    plt.show()

## 5. Validation Strategy

The main validation uses `StratifiedKFold` on target quantile bins. This preserves the target-tail structure in every fold, which is essential because RMSE is sensitive to extreme values.

A secondary `GroupKFold` by `source_id` is included as a diagnostic for source generalization. It is not automatically used as final validation because it may be much more pessimistic than the public/private leaderboard split.

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)


def create_target_bins(y, q=10):
    """Quantile bins for stratification. Falls back gracefully if duplicates occur."""
    bins = pd.qcut(y, q=q, labels=False, duplicates="drop")
    return bins.astype(int)


def create_tail_bins(y, low_q=0.05, high_q=0.95):
    q05 = y.quantile(low_q)
    q20 = y.quantile(0.20)
    q80 = y.quantile(0.80)
    q95 = y.quantile(high_q)
    labels = pd.Series("mid", index=y.index, dtype="object")
    labels[y <= q05] = "very_low"
    labels[(y > q05) & (y <= q20)] = "low"
    labels[(y >= q80) & (y < q95)] = "high"
    labels[y >= q95] = "very_high"
    return labels


def make_stratified_folds(y, n_splits=N_SPLITS, q=10, seed=SEED):
    y_binned = create_target_bins(y, q=q)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = np.zeros(len(y), dtype=int)
    for fold, (_, valid_idx) in enumerate(skf.split(np.zeros(len(y)), y_binned)):
        folds[valid_idx] = fold
    return folds, y_binned

folds, y_binned = make_stratified_folds(y, n_splits=N_SPLITS, q=10, seed=SEED)
train_fe["fold"] = folds

print(pd.Series(folds).value_counts().sort_index())
print("Target-bin distribution by fold:")
display(pd.crosstab(pd.Series(folds, name="fold"), pd.Series(y_binned, name="target_quantile_bin"), normalize="index"))

tail_bins = create_tail_bins(y)
print("Tail-bin counts:")
display(tail_bins.value_counts())

In [ ]:
def evaluate_oof(y_true, pred, name="model", folds=None, extra_df=None, verbose=True):
    pred = np.asarray(pred)
    results = []
    overall = rmse(y_true, pred)
    if verbose:
        print(f"{name} overall RMSE: {overall:.6f}")
    results.append({"model": name, "segment": "overall", "value": "all", "rmse": overall, "bias": float(np.mean(pred - y_true)), "n": len(y_true)})

    if folds is not None:
        for fold in sorted(np.unique(folds)):
            idx = folds == fold
            fold_rmse = rmse(y_true[idx], pred[idx])
            if verbose:
                print(f"  Fold {fold}: {fold_rmse:.6f}")
            results.append({"model": name, "segment": "fold", "value": fold, "rmse": fold_rmse, "bias": float(np.mean(pred[idx] - y_true[idx])), "n": int(idx.sum())})

    tb = create_tail_bins(pd.Series(y_true).reset_index(drop=True))
    for b in ["very_low", "low", "mid", "high", "very_high"]:
        idx = tb.values == b
        if idx.sum() > 0:
            results.append({"model": name, "segment": "target_bin", "value": b, "rmse": rmse(y_true[idx], pred[idx]), "bias": float(np.mean(pred[idx] - y_true[idx])), "n": int(idx.sum())})

    if extra_df is not None:
        for group_col in ["source_id", "has_band_B_spectrum"]:
            if group_col in extra_df.columns:
                temp = pd.DataFrame({"group": extra_df[group_col].fillna("Missing").astype(str), "y": y_true, "pred": pred})
                for group, part in temp.groupby("group"):
                    if len(part) >= 20:
                        results.append({
                            "model": name,
                            "segment": group_col,
                            "value": group,
                            "rmse": rmse(part["y"], part["pred"]),
                            "bias": float(np.mean(part["pred"] - part["y"])),
                            "n": len(part),
                        })
        # Missingness groups.
        if "missing_ratio" in extra_df.columns:
            miss_bin = pd.qcut(extra_df["missing_ratio"].rank(method="first"), q=4, labels=["low_missing", "mid_low_missing", "mid_high_missing", "high_missing"])
            temp = pd.DataFrame({"group": miss_bin.astype(str), "y": y_true, "pred": pred})
            for group, part in temp.groupby("group"):
                results.append({"model": name, "segment": "missingness_group", "value": group, "rmse": rmse(part["y"], part["pred"]), "bias": float(np.mean(part["pred"] - part["y"])), "n": len(part)})

    return pd.DataFrame(results)


def make_error_analysis_df(train_features, y_true, pred, name="model"):
    out = pd.DataFrame({
        ID_COL: train_features[ID_COL].values if ID_COL in train_features.columns else np.arange(len(y_true)),
        "actual": y_true.values if hasattr(y_true, "values") else y_true,
        "prediction": pred,
    })
    out["residual"] = out["prediction"] - out["actual"]
    out["abs_error"] = out["residual"].abs()
    out["squared_error"] = out["residual"] ** 2
    out["target_bin"] = create_tail_bins(pd.Series(out["actual"])).values
    for col in ["source_id", "biome", "has_band_B_spectrum"]:
        if col in train_features.columns:
            out[col] = train_features[col].values
    out["model"] = name
    return out

In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [ ]:
# Source-aware diagnostic split.
if RUN_GROUP_DIAGNOSTIC and "source_id" in train_fe.columns:
    print("GroupKFold by source_id diagnostic fold sizes:")
    groups = train_fe["source_id"].fillna("Missing").astype(str).values
    gkf = GroupKFold(n_splits=min(N_SPLITS, pd.Series(groups).nunique()))
    gfolds = np.zeros(len(y), dtype=int)
    for fold, (_, valid_idx) in enumerate(gkf.split(train_fe, y, groups)):
        gfolds[valid_idx] = fold
    display(pd.DataFrame({"fold": gfolds, "source_id": groups}).groupby("fold").agg(n=("source_id", "size"), sources=("source_id", "nunique")))
    print("Use this for robustness diagnostics, not necessarily final leaderboard selection.")

## 6. Supervised Learning

The models below all use the exact same fold assignment. Each model returns out-of-fold predictions and averaged test predictions.

In [ ]:
# Final feature lists after engineering.
DROP_COLS = [ID_COL, "fold"]
feature_cols = [c for c in train_fe.columns if c not in DROP_COLS]

# Categorical columns after feature engineering.
engineered_categorical_cols = []
for col in train_fe.columns:
    if col in DROP_COLS:
        continue
    if train_fe[col].dtype == "object" or str(train_fe[col].dtype) == "category":
        engineered_categorical_cols.append(col)

engineered_numeric_cols = [c for c in feature_cols if c not in engineered_categorical_cols]

print("Total features:", len(feature_cols))
print("Categorical features:", len(engineered_categorical_cols))
print(engineered_categorical_cols[:40])
print("Numeric features:", len(engineered_numeric_cols))
print(engineered_numeric_cols[:40])

# Keep test in the same column order as train features.
missing_cols_in_test_fe = sorted(set(feature_cols) - set(test_fe.columns))
extra_cols_in_test_fe = sorted(set(test_fe.columns) - set(feature_cols) - {ID_COL})
print("Missing columns in test_fe:", missing_cols_in_test_fe)
print("Extra columns in test_fe not used:", extra_cols_in_test_fe[:20])
assert not missing_cols_in_test_fe

In [ ]:
def transform_target(y_values, mode="identity"):
    y_values = np.asarray(y_values)
    if mode == "identity":
        return y_values
    if mode == "log1p":
        return np.log1p(np.clip(y_values, a_min=0, a_max=None))
    raise ValueError(f"Unknown target transform: {mode}")


def inverse_transform_target(pred_values, mode="identity"):
    pred_values = np.asarray(pred_values)
    if mode == "identity":
        return pred_values
    if mode == "log1p":
        return np.expm1(pred_values)
    raise ValueError(f"Unknown target transform: {mode}")


def make_tail_sample_weight(y_values, low_q=0.10, high_q=0.90, tail_weight=1.5):
    y_values = pd.Series(y_values).reset_index(drop=True)
    low_thr = y_values.quantile(low_q)
    high_thr = y_values.quantile(high_q)
    w = np.ones(len(y_values), dtype=float)
    w[(y_values <= low_thr) | (y_values >= high_thr)] = tail_weight
    return w


def add_kfold_target_encoding(train_df, test_df, y_values, folds, cols, smoothing=20.0, noise=0.0):
    """Fold-safe target encoding with smoothing.

    For each validation fold, categories are encoded using only the corresponding training folds.
    Test encodings are fitted once on full training data.
    """
    train_out = train_df.copy()
    test_out = test_df.copy()
    y_series = pd.Series(y_values, index=train_out.index)
    global_mean = y_series.mean()

    for col in cols:
        if col not in train_out.columns or col not in test_out.columns:
            continue
        new_col = f"{col}_te"
        train_out[new_col] = global_mean

        for fold in sorted(np.unique(folds)):
            tr_idx = folds != fold
            va_idx = folds == fold
            stats_df = pd.DataFrame({
                col: train_out.loc[tr_idx, col].fillna("Missing").astype(str),
                "target": y_series.loc[tr_idx]
            }).groupby(col)["target"].agg(["mean", "count"])
            smooth = (stats_df["mean"] * stats_df["count"] + global_mean * smoothing) / (stats_df["count"] + smoothing)
            mapped = train_out.loc[va_idx, col].fillna("Missing").astype(str).map(smooth).fillna(global_mean)
            if noise > 0:
                mapped = mapped * (1 + np.random.normal(0, noise, size=len(mapped)))
            train_out.loc[va_idx, new_col] = mapped.values

        full_stats = pd.DataFrame({
            col: train_out[col].fillna("Missing").astype(str),
            "target": y_series,
        }).groupby(col)["target"].agg(["mean", "count"])
        full_smooth = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
        test_out[new_col] = test_out[col].fillna("Missing").astype(str).map(full_smooth).fillna(global_mean).values

    return train_out, test_out

target_encode_cols = [c for c in [
    "source_id",
    "geo_zone_macro",
    "geo_zone_meso",
    "geo_zone_micro",
    "biome",
    "land_cover_type",
    "parent_rock_type",
    "source_biome",
    "source_geo_macro",
    "biome_land_cover",
    "parent_rock_biome",
] if c in train_fe.columns]

train_model, test_model = add_kfold_target_encoding(
    train_fe.drop(columns=["fold"]),
    test_fe,
    y,
    folds,
    target_encode_cols,
    smoothing=30.0,
    noise=0.0,
)

# Reattach fold for convenience.
train_model["fold"] = folds

model_feature_cols = [c for c in train_model.columns if c not in [ID_COL, "fold"]]
model_cat_cols = [c for c in model_feature_cols if train_model[c].dtype == "object" or str(train_model[c].dtype) == "category"]
model_num_cols = [c for c in model_feature_cols if c not in model_cat_cols]

print("Target-encoded columns:", [f"{c}_te" for c in target_encode_cols])
print("Model features:", len(model_feature_cols), "categorical:", len(model_cat_cols), "numeric:", len(model_num_cols))

In [ ]:
def make_encoded_matrices(X_train, X_valid, X_test, categorical_cols, numeric_cols):
    """Fold-safe numeric imputation and categorical ordinal encoding for LGBM/XGB."""
    Xtr_num = X_train[numeric_cols].copy()
    Xva_num = X_valid[numeric_cols].copy()
    Xte_num = X_test[numeric_cols].copy()

    imputer = SimpleImputer(strategy="median")
    Xtr_num_imp = imputer.fit_transform(Xtr_num)
    Xva_num_imp = imputer.transform(Xva_num)
    Xte_num_imp = imputer.transform(Xte_num)

    if categorical_cols:
        Xtr_cat = X_train[categorical_cols].fillna("Missing").astype(str)
        Xva_cat = X_valid[categorical_cols].fillna("Missing").astype(str)
        Xte_cat = X_test[categorical_cols].fillna("Missing").astype(str)
        encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
        Xtr_cat_enc = encoder.fit_transform(Xtr_cat)
        Xva_cat_enc = encoder.transform(Xva_cat)
        Xte_cat_enc = encoder.transform(Xte_cat)
        Xtr = np.hstack([Xtr_num_imp, Xtr_cat_enc])
        Xva = np.hstack([Xva_num_imp, Xva_cat_enc])
        Xte = np.hstack([Xte_num_imp, Xte_cat_enc])
        feature_names = numeric_cols + categorical_cols
    else:
        Xtr, Xva, Xte = Xtr_num_imp, Xva_num_imp, Xte_num_imp
        feature_names = numeric_cols

    return Xtr, Xva, Xte, feature_names


def clean_for_catboost(df, categorical_cols):
    out = df.copy()
    for col in categorical_cols:
        if col in out.columns:
            out[col] = out[col].fillna("Missing").astype(str)
    return out

In [ ]:
def train_catboost_cv(X, y_values, X_test, folds, categorical_cols, params=None, target_mode="identity", sample_weight=None, model_name="catboost"):
    if not CATBOOST_AVAILABLE:
        print("Skipping CatBoost because it is not available.")
        return None, None, None

    if params is None:
        params = {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "iterations": 2000,
            "learning_rate": 0.025,
            "depth": 7,
            "l2_leaf_reg": 8.0,
            "random_strength": 0.6,
            "bagging_temperature": 0.4,
            "border_count": 254,
            "od_type": "Iter",
            "od_wait": 250,
            "verbose": 250,
            "random_seed": SEED,
            "allow_writing_files": False,
            "thread_count": -1,
        }

    X = clean_for_catboost(X, categorical_cols)
    X_test = clean_for_catboost(X_test, categorical_cols)
    cat_cols_existing = [c for c in categorical_cols if c in X.columns]

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    models = []
    fold_scores = []

    y_arr = np.asarray(y_values)
    y_trans = transform_target(y_arr, target_mode)

    for fold in sorted(np.unique(folds)):
        tr_idx = folds != fold
        va_idx = folds == fold
        print(f"\n[{model_name}] Fold {fold}")

        X_tr = X.loc[tr_idx, :]
        X_va = X.loc[va_idx, :]
        y_tr = y_trans[tr_idx]
        y_va = y_trans[va_idx]

        train_pool = Pool(
            X_tr,
            y_tr,
            cat_features=cat_cols_existing,
            weight=None if sample_weight is None else sample_weight[tr_idx],
        )
        valid_pool = Pool(X_va, y_va, cat_features=cat_cols_existing)

        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

        va_pred_trans = model.predict(X_va)
        va_pred = inverse_transform_target(va_pred_trans, target_mode)
        oof[va_idx] = va_pred
        test_pred += inverse_transform_target(model.predict(X_test), target_mode) / len(np.unique(folds))

        fold_rmse = rmse(y_arr[va_idx], va_pred)
        fold_scores.append(fold_rmse)
        print(f"[{model_name}] Fold {fold} RMSE: {fold_rmse:.6f}")
        models.append(model)
        gc.collect()

    print(f"[{model_name}] CV RMSE: {rmse(y_arr, oof):.6f} | mean {np.mean(fold_scores):.6f} +/- {np.std(fold_scores):.6f}")
    return oof, test_pred, models

In [ ]:
def train_lgbm_cv(X, y_values, X_test, folds, categorical_cols, numeric_cols, params=None, target_mode="identity", sample_weight=None, model_name="lgbm"):
    if not LGBM_AVAILABLE:
        print("Skipping LightGBM because it is not available.")
        return None, None, None, None

    if params is None:
        params = {
            "objective": "regression",
            "metric": "rmse",
            "n_estimators": 2000,
            "learning_rate": 0.025,
            "num_leaves": 96,
            "max_depth": -1,
            "min_child_samples": 35,
            "subsample": 0.85,
            "subsample_freq": 1,
            "colsample_bytree": 0.80,
            "reg_alpha": 0.05,
            "reg_lambda": 1.5,
            "random_state": SEED,
            "n_jobs": -1,
            "verbosity": -1,
            "force_col_wise": True,
        }

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    models = []
    importances = []
    fold_scores = []
    y_arr = np.asarray(y_values)
    y_trans = transform_target(y_arr, target_mode)

    for fold in sorted(np.unique(folds)):
        tr_idx = folds != fold
        va_idx = folds == fold
        print(f"\n[{model_name}] Fold {fold}")

        X_tr, X_va, X_te, feature_names = make_encoded_matrices(
            X.loc[tr_idx, :], X.loc[va_idx, :], X_test,
            categorical_cols=categorical_cols,
            numeric_cols=numeric_cols,
        )
        y_tr = y_trans[tr_idx]
        y_va = y_trans[va_idx]

        model = LGBMRegressor(**params)
        callbacks = [lgb.early_stopping(stopping_rounds=250, verbose=False), lgb.log_evaluation(period=250)]
        model.fit(
            X_tr,
            y_tr,
            sample_weight=None if sample_weight is None else sample_weight[tr_idx],
            eval_set=[(X_va, y_va)],
            eval_metric="rmse",
            callbacks=callbacks,
        )

        va_pred = inverse_transform_target(model.predict(X_va, num_iteration=model.best_iteration_), target_mode)
        oof[va_idx] = va_pred
        test_pred += inverse_transform_target(model.predict(X_te, num_iteration=model.best_iteration_), target_mode) / len(np.unique(folds))

        fold_rmse = rmse(y_arr[va_idx], va_pred)
        fold_scores.append(fold_rmse)
        print(f"[{model_name}] Fold {fold} RMSE: {fold_rmse:.6f}")

        imp = pd.DataFrame({"feature": feature_names, "importance": model.feature_importances_, "fold": fold, "model": model_name})
        importances.append(imp)
        models.append(model)
        gc.collect()

    importance_df = pd.concat(importances, ignore_index=True) if importances else pd.DataFrame()
    print(f"[{model_name}] CV RMSE: {rmse(y_arr, oof):.6f} | mean {np.mean(fold_scores):.6f} +/- {np.std(fold_scores):.6f}")
    return oof, test_pred, models, importance_df

In [ ]:
def train_xgb_cv(X, y_values, X_test, folds, categorical_cols, numeric_cols, params=None, target_mode="identity", sample_weight=None, model_name="xgb"):
    if not XGB_AVAILABLE:
        print("Skipping XGBoost because it is not available.")
        return None, None, None, None

    if params is None:
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "n_estimators": 2000,
            "learning_rate": 0.025,
            "max_depth": 6,
            "min_child_weight": 8,
            "subsample": 0.85,
            "colsample_bytree": 0.82,
            "reg_alpha": 0.05,
            "reg_lambda": 1.8,
            "gamma": 0.02,
            "tree_method": "hist",
            "random_state": SEED,
            "n_jobs": -1,
        }

    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    models = []
    importances = []
    fold_scores = []
    y_arr = np.asarray(y_values)
    y_trans = transform_target(y_arr, target_mode)

    for fold in sorted(np.unique(folds)):
        tr_idx = folds != fold
        va_idx = folds == fold
        print(f"\n[{model_name}] Fold {fold}")

        X_tr, X_va, X_te, feature_names = make_encoded_matrices(
            X.loc[tr_idx, :], X.loc[va_idx, :], X_test,
            categorical_cols=categorical_cols,
            numeric_cols=numeric_cols,
        )
        y_tr = y_trans[tr_idx]
        y_va = y_trans[va_idx]

        model = XGBRegressor(**params)
        try:
            model.fit(
                X_tr,
                y_tr,
                sample_weight=None if sample_weight is None else sample_weight[tr_idx],
                eval_set=[(X_va, y_va)],
                verbose=250,
                early_stopping_rounds=250,
            )
        except TypeError:
            # Some newer/older xgboost versions move early stopping into constructor/callbacks.
            print("early_stopping_rounds not accepted by this XGBoost version; fitting without early stopping.")
            model.fit(
                X_tr,
                y_tr,
                sample_weight=None if sample_weight is None else sample_weight[tr_idx],
                eval_set=[(X_va, y_va)],
                verbose=250,
            )

        best_ntree_limit = getattr(model, "best_iteration", None)
        if best_ntree_limit is None:
            va_pred_trans = model.predict(X_va)
            te_pred_trans = model.predict(X_te)
        else:
            va_pred_trans = model.predict(X_va, iteration_range=(0, best_ntree_limit + 1))
            te_pred_trans = model.predict(X_te, iteration_range=(0, best_ntree_limit + 1))

        va_pred = inverse_transform_target(va_pred_trans, target_mode)
        oof[va_idx] = va_pred
        test_pred += inverse_transform_target(te_pred_trans, target_mode) / len(np.unique(folds))

        fold_rmse = rmse(y_arr[va_idx], va_pred)
        fold_scores.append(fold_rmse)
        print(f"[{model_name}] Fold {fold} RMSE: {fold_rmse:.6f}")

        imp_values = getattr(model, "feature_importances_", np.zeros(len(feature_names)))
        imp = pd.DataFrame({"feature": feature_names, "importance": imp_values, "fold": fold, "model": model_name})
        importances.append(imp)
        models.append(model)
        gc.collect()

    importance_df = pd.concat(importances, ignore_index=True) if importances else pd.DataFrame()
    print(f"[{model_name}] CV RMSE: {rmse(y_arr, oof):.6f} | mean {np.mean(fold_scores):.6f} +/- {np.std(fold_scores):.6f}")
    return oof, test_pred, models, importance_df

In [ ]:
# Train main models on original target.
X_train_model = train_model[model_feature_cols].copy()
X_test_model = test_model[model_feature_cols].copy()

oof_predictions = {}
test_predictions = {}
model_scores = []
feature_importance_parts = []

cat_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 2000,
    "learning_rate": 0.025,
    "depth": 7,
    "l2_leaf_reg": 8.0,
    "random_strength": 0.6,
    "bagging_temperature": 0.4,
    "border_count": 254,
    "od_type": "Iter",
    "od_wait": 250,
    "verbose": 250,
    "random_seed": SEED,
    "allow_writing_files": False,
    "thread_count": -1,
}

lgb_params = {
    "objective": "regression",
    "metric": "rmse",
    "n_estimators": 2000,
    "learning_rate": 0.025,
    "num_leaves": 96,
    "max_depth": -1,
    "min_child_samples": 35,
    "subsample": 0.85,
    "subsample_freq": 1,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.05,
    "reg_lambda": 1.5,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
}

xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "n_estimators": 2000,
    "learning_rate": 0.025,
    "max_depth": 6,
    "min_child_weight": 8,
    "subsample": 0.85,
    "colsample_bytree": 0.82,
    "reg_alpha": 0.05,
    "reg_lambda": 1.8,
    "gamma": 0.02,
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": -1,
}

cat_oof, cat_test, cat_models = train_catboost_cv(
    X_train_model, y.values, X_test_model, folds,
    categorical_cols=model_cat_cols,
    params=cat_params,
    target_mode="identity",
    model_name="catboost",
)
if cat_oof is not None:
    oof_predictions["catboost"] = cat_oof
    test_predictions["catboost"] = cat_test
    model_scores.append(evaluate_oof(y.values, cat_oof, "catboost", folds, train_model, verbose=False))

lgb_oof, lgb_test, lgb_models, lgb_importance = train_lgbm_cv(
    X_train_model, y.values, X_test_model, folds,
    categorical_cols=model_cat_cols,
    numeric_cols=model_num_cols,
    params=lgb_params,
    target_mode="identity",
    model_name="lgbm",
)
if lgb_oof is not None:
    oof_predictions["lgbm"] = lgb_oof
    test_predictions["lgbm"] = lgb_test
    model_scores.append(evaluate_oof(y.values, lgb_oof, "lgbm", folds, train_model, verbose=False))
    feature_importance_parts.append(lgb_importance)

xgb_oof, xgb_test, xgb_models, xgb_importance = train_xgb_cv(
    X_train_model, y.values, X_test_model, folds,
    categorical_cols=model_cat_cols,
    numeric_cols=model_num_cols,
    params=xgb_params,
    target_mode="identity",
    model_name="xgb",
)
if xgb_oof is not None:
    oof_predictions["xgb"] = xgb_oof
    test_predictions["xgb"] = xgb_test
    model_scores.append(evaluate_oof(y.values, xgb_oof, "xgb", folds, train_model, verbose=False))
    feature_importance_parts.append(xgb_importance)

scores_df = pd.concat(model_scores, ignore_index=True) if model_scores else pd.DataFrame()
display(scores_df[scores_df["segment"].isin(["overall", "target_bin", "fold"])] if not scores_df.empty else scores_df)

In [ ]:
# Linear baseline with one-hot encoding. Useful sanity check, usually not final.
def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, min_frequency=5)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


def train_ridge_baseline_cv(X, y_values, X_test, folds, categorical_cols, numeric_cols, alpha=20.0):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    y_arr = np.asarray(y_values)

    for fold in sorted(np.unique(folds)):
        tr_idx = folds != fold
        va_idx = folds == fold
        pre = ColumnTransformer(
            transformers=[
                ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
                ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Missing")), ("onehot", make_onehot_encoder())]), categorical_cols),
            ],
            remainder="drop",
        )
        model = Pipeline([
            ("pre", pre),
            ("ridge", Ridge(alpha=alpha, random_state=SEED)),
        ])
        model.fit(X.loc[tr_idx], y_arr[tr_idx])
        oof[va_idx] = model.predict(X.loc[va_idx])
        test_pred += model.predict(X_test) / len(np.unique(folds))
    print(f"ridge_baseline CV RMSE: {rmse(y_arr, oof):.6f}")
    return oof, test_pred

ridge_oof, ridge_test = train_ridge_baseline_cv(X_train_model, y.values, X_test_model, folds, model_cat_cols, model_num_cols)
oof_predictions["ridge"] = ridge_oof
test_predictions["ridge"] = ridge_test
model_scores.append(evaluate_oof(y.values, ridge_oof, "ridge", folds, train_model, verbose=False))

## 7. Unsupervised Learning + Hyperparameter Tuning

Cluster and PCA features have already been added above as target-independent feature engineering. The cells below provide modest manual/Optuna tuning for the manually defined models. Keep `RUN_OPTUNA=False` for a faster full notebook run; set it to `True` for tuning.

In [ ]:
def tune_lgbm_optuna(X, y_values, folds, categorical_cols, numeric_cols, n_trials=20):
    if not (OPTUNA_AVAILABLE and LGBM_AVAILABLE):
        print("Optuna or LightGBM unavailable; skipping LGBM tuning.")
        return None

    def objective(trial):
        params = {
            "objective": "regression",
            "metric": "rmse",
            "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.06, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 32, 192),
            "max_depth": trial.suggest_int("max_depth", 3, 12),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 120),
            "subsample": trial.suggest_float("subsample", 0.60, 1.00),
            "subsample_freq": 1,
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.00),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 8.0, log=True),
            "random_state": SEED,
            "n_jobs": -1,
            "verbosity": -1,
            "force_col_wise": True,
        }
        # Use first 3 folds for faster tuning; final models still use all folds.
        fold_scores = []
        for fold in sorted(np.unique(folds))[:min(3, len(np.unique(folds)))]:
            tr_idx = folds != fold
            va_idx = folds == fold
            X_tr, X_va, _, _ = make_encoded_matrices(
                X.loc[tr_idx], X.loc[va_idx], X.iloc[:1].copy(), categorical_cols, numeric_cols
            )
            model = LGBMRegressor(**params)
            model.fit(
                X_tr, y_values[tr_idx],
                eval_set=[(X_va, y_values[va_idx])],
                eval_metric="rmse",
                callbacks=[lgb.early_stopping(200, verbose=False)],
            )
            pred = model.predict(X_va, num_iteration=model.best_iteration_)
            fold_scores.append(rmse(y_values[va_idx], pred))
        return float(np.mean(fold_scores))

    study = optuna.create_study(direction="minimize", study_name="lgbm_rmse")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print("Best LGBM RMSE:", study.best_value)
    print("Best LGBM params:", study.best_params)
    return study.best_params


def tune_catboost_optuna(X, y_values, folds, categorical_cols, n_trials=20):
    if not (OPTUNA_AVAILABLE and CATBOOST_AVAILABLE):
        print("Optuna or CatBoost unavailable; skipping CatBoost tuning.")
        return None

    X_cb = clean_for_catboost(X, categorical_cols)
    cat_cols_existing = [c for c in categorical_cols if c in X_cb.columns]

    def objective(trial):
        params = {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "iterations": trial.suggest_int("iterations", 500, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.06, log=True),
            "depth": trial.suggest_int("depth", 4, 9),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 15.0),
            "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
            "border_count": trial.suggest_int("border_count", 64, 254),
            "od_type": "Iter",
            "od_wait": 200,
            "verbose": False,
            "random_seed": SEED,
            "allow_writing_files": False,
            "thread_count": -1,
        }
        fold_scores = []
        for fold in sorted(np.unique(folds))[:min(3, len(np.unique(folds)))]:
            tr_idx = folds != fold
            va_idx = folds == fold
            model = CatBoostRegressor(**params)
            model.fit(
                Pool(X_cb.loc[tr_idx], y_values[tr_idx], cat_features=cat_cols_existing),
                eval_set=Pool(X_cb.loc[va_idx], y_values[va_idx], cat_features=cat_cols_existing),
                use_best_model=True,
            )
            pred = model.predict(X_cb.loc[va_idx])
            fold_scores.append(rmse(y_values[va_idx], pred))
        return float(np.mean(fold_scores))

    study = optuna.create_study(direction="minimize", study_name="catboost_rmse")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print("Best CatBoost RMSE:", study.best_value)
    print("Best CatBoost params:", study.best_params)
    return study.best_params


def tune_xgb_optuna(X, y_values, folds, categorical_cols, numeric_cols, n_trials=20):
    if not (OPTUNA_AVAILABLE and XGB_AVAILABLE):
        print("Optuna or XGBoost unavailable; skipping XGB tuning.")
        return None

    def objective(trial):
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "n_estimators": trial.suggest_int("n_estimators", 500, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.06, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
            "subsample": trial.suggest_float("subsample", 0.60, 1.00),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.00),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 8.0, log=True),
            "gamma": trial.suggest_float("gamma", 1e-4, 2.0, log=True),
            "tree_method": "hist",
            "random_state": SEED,
            "n_jobs": -1,
        }
        fold_scores = []
        for fold in sorted(np.unique(folds))[:min(3, len(np.unique(folds)))]:
            tr_idx = folds != fold
            va_idx = folds == fold
            X_tr, X_va, _, _ = make_encoded_matrices(
                X.loc[tr_idx], X.loc[va_idx], X.iloc[:1].copy(), categorical_cols, numeric_cols
            )
            model = XGBRegressor(**params)
            try:
                model.fit(X_tr, y_values[tr_idx], eval_set=[(X_va, y_values[va_idx])], verbose=False, early_stopping_rounds=200)
            except TypeError:
                model.fit(X_tr, y_values[tr_idx], eval_set=[(X_va, y_values[va_idx])], verbose=False)
            pred = model.predict(X_va)
            fold_scores.append(rmse(y_values[va_idx], pred))
        return float(np.mean(fold_scores))

    study = optuna.create_study(direction="minimize", study_name="xgb_rmse")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print("Best XGB RMSE:", study.best_value)
    print("Best XGB params:", study.best_params)
    return study.best_params

if RUN_OPTUNA:
    best_lgb_params = tune_lgbm_optuna(X_train_model, y.values, folds, model_cat_cols, model_num_cols, N_OPTUNA_TRIALS)
    best_cat_params = tune_catboost_optuna(X_train_model, y.values, folds, model_cat_cols, N_OPTUNA_TRIALS)
    best_xgb_params = tune_xgb_optuna(X_train_model, y.values, folds, model_cat_cols, model_num_cols, N_OPTUNA_TRIALS)
else:
    print("RUN_OPTUNA=False. Using manually selected robust parameters above.")

## 8. Tail-Aware Modeling for Very Low and Very High Target Values

This section directly attacks the baseline weakness: regression-to-the-mean in the target tails. Every technique is compared using OOF predictions and is only used in the final ensemble if it improves global RMSE or tail stability.

In [ ]:
def tail_report_from_dict(pred_dict, y_true, folds, features):
    reports = []
    for name, pred in pred_dict.items():
        reports.append(evaluate_oof(y_true, pred, name, folds, features, verbose=False))
    return pd.concat(reports, ignore_index=True) if reports else pd.DataFrame()

current_report = tail_report_from_dict(oof_predictions, y.values, folds, train_model)
display(current_report[current_report["segment"].isin(["overall", "target_bin"])] if not current_report.empty else current_report)

### A. Target transformation experiment

`log1p(target)` may reduce high-tail dominance, but it can also worsen original-scale RMSE because the competition metric is not log-RMSE. We test it rather than assuming it wins.

In [ ]:
if RUN_LOG_TARGET_EXPERIMENT:
    log_oof_predictions = {}
    log_test_predictions = {}

    cat_log_oof, cat_log_test, _ = train_catboost_cv(
        X_train_model, y.values, X_test_model, folds,
        categorical_cols=model_cat_cols,
        params=cat_params,
        target_mode="log1p",
        model_name="catboost_log1p",
    )
    if cat_log_oof is not None:
        log_oof_predictions["catboost_log1p"] = cat_log_oof
        log_test_predictions["catboost_log1p"] = cat_log_test

    lgb_log_oof, lgb_log_test, _, lgb_log_importance = train_lgbm_cv(
        X_train_model, y.values, X_test_model, folds,
        categorical_cols=model_cat_cols,
        numeric_cols=model_num_cols,
        params=lgb_params,
        target_mode="log1p",
        model_name="lgbm_log1p",
    )
    if lgb_log_oof is not None:
        log_oof_predictions["lgbm_log1p"] = lgb_log_oof
        log_test_predictions["lgbm_log1p"] = lgb_log_test
        feature_importance_parts.append(lgb_log_importance)

    xgb_log_oof, xgb_log_test, _, xgb_log_importance = train_xgb_cv(
        X_train_model, y.values, X_test_model, folds,
        categorical_cols=model_cat_cols,
        numeric_cols=model_num_cols,
        params=xgb_params,
        target_mode="log1p",
        model_name="xgb_log1p",
    )
    if xgb_log_oof is not None:
        log_oof_predictions["xgb_log1p"] = xgb_log_oof
        log_test_predictions["xgb_log1p"] = xgb_log_test
        feature_importance_parts.append(xgb_log_importance)

    # Keep all variants available for ensemble selection.
    oof_predictions.update(log_oof_predictions)
    test_predictions.update(log_test_predictions)

    log_report = tail_report_from_dict(log_oof_predictions, y.values, folds, train_model)
    display(log_report[log_report["segment"].isin(["overall", "target_bin"])] if not log_report.empty else log_report)
else:
    print("RUN_LOG_TARGET_EXPERIMENT=False")

### B. Weighted training experiment

Tail samples receive modest extra weight. The weight is intentionally conservative because over-weighting tails can harm the central mass enough to lose overall RMSE.

In [ ]:
if RUN_WEIGHTED_EXPERIMENT:
    tail_weights = make_tail_sample_weight(y.values, low_q=0.10, high_q=0.90, tail_weight=1.5)
    print(pd.Series(tail_weights).value_counts())

    weighted_oof_predictions = {}
    weighted_test_predictions = {}

    # Weighted LightGBM is usually the fastest useful tail experiment.
    lgb_w_oof, lgb_w_test, _, lgb_w_importance = train_lgbm_cv(
        X_train_model, y.values, X_test_model, folds,
        categorical_cols=model_cat_cols,
        numeric_cols=model_num_cols,
        params=lgb_params,
        target_mode="identity",
        sample_weight=tail_weights,
        model_name="lgbm_tail_weighted",
    )
    if lgb_w_oof is not None:
        weighted_oof_predictions["lgbm_tail_weighted"] = lgb_w_oof
        weighted_test_predictions["lgbm_tail_weighted"] = lgb_w_test
        feature_importance_parts.append(lgb_w_importance)

    # Weighted CatBoost, if available.
    cat_w_oof, cat_w_test, _ = train_catboost_cv(
        X_train_model, y.values, X_test_model, folds,
        categorical_cols=model_cat_cols,
        params=cat_params,
        target_mode="identity",
        sample_weight=tail_weights,
        model_name="catboost_tail_weighted",
    )
    if cat_w_oof is not None:
        weighted_oof_predictions["catboost_tail_weighted"] = cat_w_oof
        weighted_test_predictions["catboost_tail_weighted"] = cat_w_test

    oof_predictions.update(weighted_oof_predictions)
    test_predictions.update(weighted_test_predictions)

    weighted_report = tail_report_from_dict(weighted_oof_predictions, y.values, folds, train_model)
    display(weighted_report[weighted_report["segment"].isin(["overall", "target_bin"])] if not weighted_report.empty else weighted_report)
else:
    print("RUN_WEIGHTED_EXPERIMENT=False")

### C. Ensemble helpers and clipping tests

In [ ]:
def simple_average(pred_dict, keys=None):
    if keys is None:
        keys = list(pred_dict.keys())
    arr = np.vstack([pred_dict[k] for k in keys])
    return arr.mean(axis=0)


def optimize_ensemble_weights(oof_dict, y_true, keys=None, n_iter=20000, seed=SEED):
    if keys is None:
        keys = list(oof_dict.keys())
    rng = np.random.default_rng(seed)
    P = np.vstack([oof_dict[k] for k in keys]).T
    best_w = np.ones(len(keys)) / len(keys)
    best_score = rmse(y_true, P @ best_w)

    # Include equal, one-hot, and random Dirichlet candidates.
    candidates = [best_w]
    candidates.extend(np.eye(len(keys)))
    for _ in range(n_iter):
        candidates.append(rng.dirichlet(np.ones(len(keys))))

    for w in candidates:
        pred = P @ w
        score = rmse(y_true, pred)
        if score < best_score:
            best_score = score
            best_w = w

    return dict(zip(keys, best_w)), best_score


def make_stacking_predictions(oof_dict, test_dict, y_true, folds, keys=None, model=None):
    if keys is None:
        keys = list(oof_dict.keys())
    if model is None:
        model = Ridge(alpha=1.0, positive=False)

    P_oof = np.vstack([oof_dict[k] for k in keys]).T
    P_test = np.vstack([test_dict[k] for k in keys]).T
    stack_oof = np.zeros(len(y_true))
    stack_test = np.zeros(len(P_test))

    for fold in sorted(np.unique(folds)):
        tr_idx = folds != fold
        va_idx = folds == fold
        m = Ridge(alpha=getattr(model, "alpha", 1.0)) if isinstance(model, Ridge) else model
        m.fit(P_oof[tr_idx], y_true[tr_idx])
        stack_oof[va_idx] = m.predict(P_oof[va_idx])
        stack_test += m.predict(P_test) / len(np.unique(folds))

    final_model = Ridge(alpha=getattr(model, "alpha", 1.0)) if isinstance(model, Ridge) else model
    final_model.fit(P_oof, y_true)
    stack_test_full = final_model.predict(P_test)
    # Use full-fit test prediction; OOF remains fold-safe.
    return stack_oof, stack_test_full, final_model


def evaluate_clipping(y_true, pred, name="pred"):
    candidates = {
        "no_clip": pred.copy(),
        "clip_train_minmax": np.clip(pred, y_true.min(), y_true.max()),
        "clip_q001_q999": np.clip(pred, np.quantile(y_true, 0.001), np.quantile(y_true, 0.999)),
        "clip_nonnegative": np.clip(pred, 0, None),
    }
    rows = []
    for clip_name, p in candidates.items():
        rep = evaluate_oof(y_true, p, f"{name}_{clip_name}", folds=None, extra_df=None, verbose=False)
        tail_rep = evaluate_oof(y_true, p, f"{name}_{clip_name}", folds=None, extra_df=None, verbose=False)
        overall = rep.query("segment == 'overall'")["rmse"].iloc[0]
        rows.append({"prediction": name, "clip_strategy": clip_name, "rmse": overall})
    return pd.DataFrame(rows), candidates

main_base_keys = [k for k in ["catboost", "lgbm", "xgb"] if k in oof_predictions]
all_model_keys = list(oof_predictions.keys())
print("Main base keys:", main_base_keys)
print("All available prediction keys:", all_model_keys)

### D. Simple average, weighted average, and stacking

In [ ]:
ensemble_oof = {}
ensemble_test = {}
ensemble_details = []

if len(main_base_keys) >= 2:
    ensemble_oof["simple_avg_main"] = simple_average(oof_predictions, main_base_keys)
    ensemble_test["simple_avg_main"] = simple_average(test_predictions, main_base_keys)

if len(all_model_keys) >= 2:
    best_weights, best_weight_score = optimize_ensemble_weights(oof_predictions, y.values, keys=all_model_keys, n_iter=20000, seed=SEED)
    weighted_pred = sum(oof_predictions[k] * w for k, w in best_weights.items())
    weighted_test = sum(test_predictions[k] * w for k, w in best_weights.items())
    ensemble_oof["weighted_avg_all"] = weighted_pred
    ensemble_test["weighted_avg_all"] = weighted_test
    ensemble_details.append({"ensemble": "weighted_avg_all", "weights": best_weights, "cv_rmse": best_weight_score})
    print("Best weighted-average RMSE:", best_weight_score)
    print("Best weights:")
    display(pd.Series(best_weights).sort_values(ascending=False).to_frame("weight"))

if len(all_model_keys) >= 2:
    stack_oof, stack_test, stack_model = make_stacking_predictions(oof_predictions, test_predictions, y.values, folds, keys=all_model_keys, model=Ridge(alpha=1.0))
    ensemble_oof["ridge_stack_all"] = stack_oof
    ensemble_test["ridge_stack_all"] = stack_test
    print("Ridge stacking RMSE:", rmse(y.values, stack_oof))

ens_report = tail_report_from_dict(ensemble_oof, y.values, folds, train_model)
display(ens_report[ens_report["segment"].isin(["overall", "target_bin"])] if not ens_report.empty else ens_report)

### E. Residual correction model

A second-stage model learns the remaining residual pattern from OOF predictions. The residual correction is validated out-of-fold to avoid training on its own target residuals.

In [ ]:
def residual_correction_ridge_cv(base_oof, base_test, X, X_test, y_values, folds, categorical_cols, numeric_cols, alpha=30.0, correction_strength=0.5):
    residual = y_values - base_oof
    X_aug = X.copy()
    X_test_aug = X_test.copy()
    X_aug["base_pred"] = base_oof
    X_test_aug["base_pred"] = base_test
    num_cols_aug = numeric_cols + ["base_pred"]
    cat_cols_aug = categorical_cols

    corr_oof = np.zeros(len(y_values))
    corr_test_accum = np.zeros(len(X_test))

    for fold in sorted(np.unique(folds)):
        tr_idx = folds != fold
        va_idx = folds == fold
        pre = ColumnTransformer(
            transformers=[
                ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols_aug),
                ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Missing")), ("onehot", make_onehot_encoder())]), cat_cols_aug),
            ],
            remainder="drop",
        )
        model = Pipeline([("pre", pre), ("ridge", Ridge(alpha=alpha, random_state=SEED))])
        model.fit(X_aug.loc[tr_idx], residual[tr_idx])
        corr_oof[va_idx] = model.predict(X_aug.loc[va_idx])
        corr_test_accum += model.predict(X_test_aug) / len(np.unique(folds))

    corrected_oof = base_oof + correction_strength * corr_oof

    # Full fit for test residual correction.
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols_aug),
            ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Missing")), ("onehot", make_onehot_encoder())]), cat_cols_aug),
        ],
        remainder="drop",
    )
    full_model = Pipeline([("pre", pre), ("ridge", Ridge(alpha=alpha, random_state=SEED))])
    full_model.fit(X_aug, residual)
    corrected_test = base_test + correction_strength * full_model.predict(X_test_aug)
    return corrected_oof, corrected_test

# Use the strongest current ensemble as base for residual correction.
if ensemble_oof:
    base_name_for_correction = min(ensemble_oof.keys(), key=lambda k: rmse(y.values, ensemble_oof[k]))
    print("Base for residual correction:", base_name_for_correction)
    res_oof, res_test = residual_correction_ridge_cv(
        ensemble_oof[base_name_for_correction], ensemble_test[base_name_for_correction],
        X_train_model, X_test_model, y.values, folds,
        model_cat_cols, model_num_cols,
        alpha=50.0,
        correction_strength=0.35,
    )
    ensemble_oof["residual_corrected"] = res_oof
    ensemble_test["residual_corrected"] = res_test
    print("Residual-corrected RMSE:", rmse(y.values, res_oof))
else:
    print("No ensemble available for residual correction.")

### F. Bin-specific calibration

True test target bins are unknown, so calibration uses predicted quantiles. This keeps the correction realistic and prevents using actual target-tail labels at inference.

In [ ]:
def bin_specific_calibration_cv(base_oof, base_test, y_values, folds, n_bins=5):
    base_oof = np.asarray(base_oof)
    base_test = np.asarray(base_test)
    calibrated_oof = np.zeros(len(y_values))

    # Predicted-value bins, not true target bins.
    pred_bins_all = pd.qcut(pd.Series(base_oof).rank(method="first"), q=n_bins, labels=False, duplicates="drop").values

    for fold in sorted(np.unique(folds)):
        tr_idx = folds != fold
        va_idx = folds == fold
        tr_pred = base_oof[tr_idx]
        va_pred = base_oof[va_idx]
        tr_y = y_values[tr_idx]
        tr_bins = pred_bins_all[tr_idx]
        va_bins = pd.cut(
            va_pred,
            bins=np.quantile(tr_pred, np.linspace(0, 1, n_bins + 1)),
            labels=False,
            include_lowest=True,
            duplicates="drop",
        )
        va_bins = pd.Series(va_bins).fillna(n_bins // 2).astype(int).values

        fold_cal = np.zeros(len(va_pred))
        global_lr = LinearRegression().fit(tr_pred.reshape(-1, 1), tr_y)
        for b in np.unique(va_bins):
            idx_tr_b = tr_bins == b
            idx_va_b = va_bins == b
            if idx_tr_b.sum() >= 50:
                lr = LinearRegression().fit(tr_pred[idx_tr_b].reshape(-1, 1), tr_y[idx_tr_b])
            else:
                lr = global_lr
            fold_cal[idx_va_b] = lr.predict(va_pred[idx_va_b].reshape(-1, 1))
        calibrated_oof[va_idx] = fold_cal

    # Test calibration from full train predicted quantiles.
    full_bins = pred_bins_all
    test_bins = pd.cut(
        base_test,
        bins=np.quantile(base_oof, np.linspace(0, 1, n_bins + 1)),
        labels=False,
        include_lowest=True,
        duplicates="drop",
    )
    test_bins = pd.Series(test_bins).fillna(n_bins // 2).astype(int).values
    calibrated_test = np.zeros(len(base_test))
    global_lr = LinearRegression().fit(base_oof.reshape(-1, 1), y_values)
    for b in np.unique(test_bins):
        idx_tr_b = full_bins == b
        idx_te_b = test_bins == b
        if idx_tr_b.sum() >= 50:
            lr = LinearRegression().fit(base_oof[idx_tr_b].reshape(-1, 1), y_values[idx_tr_b])
        else:
            lr = global_lr
        calibrated_test[idx_te_b] = lr.predict(base_test[idx_te_b].reshape(-1, 1))
    return calibrated_oof, calibrated_test

if ensemble_oof:
    base_name_for_cal = min(ensemble_oof.keys(), key=lambda k: rmse(y.values, ensemble_oof[k]))
    print("Base for bin-specific calibration:", base_name_for_cal)
    cal_oof, cal_test = bin_specific_calibration_cv(ensemble_oof[base_name_for_cal], ensemble_test[base_name_for_cal], y.values, folds, n_bins=5)
    ensemble_oof["bin_calibrated"] = cal_oof
    ensemble_test["bin_calibrated"] = cal_test
    print("Bin-calibrated RMSE:", rmse(y.values, cal_oof))
else:
    print("No ensemble available for calibration.")

In [ ]:
# Clipping strategy comparison for all candidate ensembles.
clip_rows = []
clip_candidates = {}
for name, pred in ensemble_oof.items():
    clip_df, candidates = evaluate_clipping(y.values, pred, name=name)
    clip_rows.append(clip_df)
    clip_candidates[name] = candidates
clip_results = pd.concat(clip_rows, ignore_index=True) if clip_rows else pd.DataFrame()
display(clip_results.sort_values("rmse") if not clip_results.empty else clip_results)

## 9. Ensemble Selection

The final model is chosen from base models and ensembles. Selection prioritizes global RMSE, but when candidates are close, tail-bin RMSE and bias are used as tie-breakers.

In [ ]:
# Collect all candidates.
candidate_oof = {}
candidate_test = {}

candidate_oof.update(oof_predictions)
candidate_test.update(test_predictions)
candidate_oof.update(ensemble_oof)
candidate_test.update(ensemble_test)

# Apply clipping candidates conservatively to ensemble candidates only.
for name, pred in list(ensemble_oof.items()):
    for clip_name, clipped_pred in clip_candidates.get(name, {}).items():
        full_name = f"{name}_{clip_name}"
        candidate_oof[full_name] = clipped_pred
        if clip_name == "no_clip":
            candidate_test[full_name] = ensemble_test[name]
        elif clip_name == "clip_train_minmax":
            candidate_test[full_name] = np.clip(ensemble_test[name], y.min(), y.max())
        elif clip_name == "clip_q001_q999":
            candidate_test[full_name] = np.clip(ensemble_test[name], y.quantile(0.001), y.quantile(0.999))
        elif clip_name == "clip_nonnegative":
            candidate_test[full_name] = np.clip(ensemble_test[name], 0, None)

# Score candidates by global and tail RMSE.
rows = []
for name, pred in candidate_oof.items():
    rep = evaluate_oof(y.values, pred, name, folds, train_model, verbose=False)
    overall = rep.query("segment == 'overall'")["rmse"].iloc[0]
    target_rep = rep.query("segment == 'target_bin'")
    very_low_rmse = target_rep.query("value == 'very_low'")["rmse"].iloc[0]
    very_high_rmse = target_rep.query("value == 'very_high'")["rmse"].iloc[0]
    tail_avg = (very_low_rmse + very_high_rmse) / 2
    rows.append({
        "candidate": name,
        "overall_rmse": overall,
        "very_low_rmse": very_low_rmse,
        "very_high_rmse": very_high_rmse,
        "tail_avg_rmse": tail_avg,
        "selection_score": overall + 0.10 * tail_avg,
    })

candidate_scores = pd.DataFrame(rows).sort_values(["overall_rmse", "tail_avg_rmse"])
display(candidate_scores.head(30))

# Select the best global RMSE, unless a near-tie has better tail stability.
best_global = candidate_scores.iloc[0]
near = candidate_scores[candidate_scores["overall_rmse"] <= best_global["overall_rmse"] * 1.005].copy()
selected = near.sort_values(["tail_avg_rmse", "overall_rmse"]).iloc[0]
FINAL_NAME = selected["candidate"]
final_oof = candidate_oof[FINAL_NAME]
final_test_pred = candidate_test[FINAL_NAME]

print("Final selected candidate:", FINAL_NAME)
print("Final CV RMSE:", rmse(y.values, final_oof))
print("Selection details:")
display(selected.to_frame().T)

final_report = evaluate_oof(y.values, final_oof, FINAL_NAME, folds, train_model, verbose=True)
display(final_report[final_report["segment"].isin(["overall", "fold", "target_bin", "has_band_B_spectrum", "missingness_group"])])

In [ ]:
# Expose required named outputs.
oof_catboost = oof_predictions.get("catboost")
oof_lgbm = oof_predictions.get("lgbm")
oof_xgb = oof_predictions.get("xgb")
oof_ensemble = final_oof

test_pred_catboost = test_predictions.get("catboost")
test_pred_lgbm = test_predictions.get("lgbm")
test_pred_xgb = test_predictions.get("xgb")
test_pred_ensemble = final_test_pred

print("Named outputs created:")
for k in ["oof_catboost", "oof_lgbm", "oof_xgb", "oof_ensemble", "test_pred_catboost", "test_pred_lgbm", "test_pred_xgb", "test_pred_ensemble"]:
    print(k, "OK" if globals().get(k) is not None else "None")

## 10. Data Visualization

The charts below summarize model quality, target-tail behavior, feature importance, and prediction distribution alignment.

In [ ]:
# Model comparison bar chart.
plt.figure(figsize=(12, 5))
sns.barplot(data=candidate_scores.head(20), x="overall_rmse", y="candidate")
plt.title("Top candidate models / ensembles by OOF RMSE")
plt.xlabel("OOF RMSE")
plt.ylabel("Candidate")
plt.show()

# RMSE by target bin for final model.
final_target_bin_report = final_report[final_report["segment"] == "target_bin"].copy()
plt.figure(figsize=(9, 4))
sns.barplot(data=final_target_bin_report, x="value", y="rmse", order=["very_low", "low", "mid", "high", "very_high"])
plt.title(f"RMSE by target bin: {FINAL_NAME}")
plt.xlabel("Target bin")
plt.ylabel("RMSE")
plt.show()

# Bias by target bin.
plt.figure(figsize=(9, 4))
sns.barplot(data=final_target_bin_report, x="value", y="bias", order=["very_low", "low", "mid", "high", "very_high"])
plt.axhline(0, color="black", linewidth=1)
plt.title(f"Bias by target bin: {FINAL_NAME}")
plt.xlabel("Target bin")
plt.ylabel("Mean(pred - actual)")
plt.show()

In [ ]:
error_df = make_error_analysis_df(train_model, y, final_oof, name=FINAL_NAME)
display(error_df.head())

plt.figure(figsize=(6, 6))
sns.scatterplot(x=error_df["actual"], y=error_df["prediction"], alpha=0.35, s=18)
lo = min(error_df["actual"].min(), error_df["prediction"].min())
hi = max(error_df["actual"].max(), error_df["prediction"].max())
plt.plot([lo, hi], [lo, hi], color="black", linewidth=1)
plt.title("Actual vs predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(error_df["residual"], bins=80, kde=True)
plt.axvline(0, color="black", linewidth=1)
plt.title("Residual distribution")
plt.xlabel("Prediction - actual")
plt.show()

plt.figure(figsize=(10, 5))
sns.scatterplot(data=error_df, x="actual", y="residual", hue="target_bin", alpha=0.45, s=18)
plt.axhline(0, color="black", linewidth=1)
plt.title("Residual vs actual by target bin")
plt.xlabel("Actual target")
plt.ylabel("Residual")
plt.legend(loc="best")
plt.show()

In [ ]:
# Error by source_id and spectral availability.
if "source_id" in error_df.columns:
    source_error = error_df.groupby("source_id").agg(
        n=("actual", "size"),
        rmse=("squared_error", lambda s: np.sqrt(np.mean(s))),
        mean_abs_error=("abs_error", "mean"),
    ).query("n >= 20").sort_values("rmse", ascending=False)
    display(source_error.head(25))

    plt.figure(figsize=(10, max(5, 0.35 * min(25, len(source_error)))))
    sns.barplot(data=source_error.head(25).reset_index(), x="rmse", y="source_id")
    plt.title("Highest RMSE source_id groups | n >= 20")
    plt.xlabel("RMSE")
    plt.ylabel("source_id")
    plt.show()

if "has_band_B_spectrum" in error_df.columns:
    band_error = error_df.groupby("has_band_B_spectrum").agg(
        n=("actual", "size"),
        rmse=("squared_error", lambda s: np.sqrt(np.mean(s))),
        mean_abs_error=("abs_error", "mean"),
    ).sort_values("rmse", ascending=False)
    display(band_error)

    plt.figure(figsize=(7, 4))
    sns.barplot(data=band_error.reset_index(), x="has_band_B_spectrum", y="rmse")
    plt.title("RMSE by band B spectrum availability")
    plt.xlabel("has_band_B_spectrum")
    plt.ylabel("RMSE")
    plt.show()

In [ ]:
# Feature importance for tree models.
feature_importance = pd.concat(feature_importance_parts, ignore_index=True) if feature_importance_parts else pd.DataFrame()
if not feature_importance.empty:
    importance_summary = feature_importance.groupby(["model", "feature"], as_index=False)["importance"].mean()
    display(importance_summary.sort_values("importance", ascending=False).head(50))

    for model_name in importance_summary["model"].unique():
        top_imp = importance_summary[importance_summary["model"] == model_name].sort_values("importance", ascending=False).head(30)
        plt.figure(figsize=(10, 8))
        sns.barplot(data=top_imp, x="importance", y="feature")
        plt.title(f"Top feature importance: {model_name}")
        plt.xlabel("Mean importance")
        plt.ylabel("Feature")
        plt.show()
else:
    importance_summary = pd.DataFrame()
    print("No feature importance available.")

In [ ]:
# CatBoost feature importance, if CatBoost model exists.
if "cat_models" in globals() and cat_models is not None and len(cat_models) > 0:
    cb_imps = []
    for fold, model in enumerate(cat_models):
        imp_vals = model.get_feature_importance()
        cb_imps.append(pd.DataFrame({"feature": X_train_model.columns, "importance": imp_vals, "fold": fold, "model": "catboost"}))
    cb_importance = pd.concat(cb_imps, ignore_index=True)
    cb_imp_summary = cb_importance.groupby("feature", as_index=False)["importance"].mean().sort_values("importance", ascending=False)
    display(cb_imp_summary.head(40))

    plt.figure(figsize=(10, 8))
    sns.barplot(data=cb_imp_summary.head(30), x="importance", y="feature")
    plt.title("Top CatBoost feature importance")
    plt.xlabel("Mean importance")
    plt.ylabel("Feature")
    plt.show()
else:
    cb_importance = pd.DataFrame()

In [ ]:
# Ensemble weight visualization.
if ensemble_details:
    w_series = pd.Series(ensemble_details[0]["weights"]).sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=w_series.values, y=w_series.index)
    plt.title("Optimized ensemble weights")
    plt.xlabel("Weight")
    plt.ylabel("Model")
    plt.show()

# Prediction distribution: train OOF vs test final.
plt.figure(figsize=(10, 5))
sns.kdeplot(final_oof, label="OOF final", fill=False)
sns.kdeplot(final_test_pred, label="Test final", fill=False)
sns.kdeplot(y.values, label="Train target", fill=False)
plt.title("Prediction distribution: OOF vs test vs actual target")
plt.xlabel(TARGET)
plt.legend()
plt.show()

## 11. Modeling Pitch

**Problem statement.** The goal is to predict soil organic content from heterogeneous soil sample metadata, chemistry, geospatial fields, and spectral PCA features. The target is continuous, and performance is measured using RMSE.

**Why soil organic content matters.** Soil organic content is associated with soil fertility, carbon storage, water retention, structure, and long-term land productivity. Better prediction can support faster soil assessment and more targeted sampling.

**Why RMSE is important.** RMSE penalizes large errors strongly, which is appropriate when large underestimates or overestimates of organic content can meaningfully distort downstream decisions. It also means rare extreme targets deserve special diagnostic attention.

**Main modeling challenge.** The dataset contains structured missingness, uneven spectral availability, source-specific sampling protocols, categorical geography effects, and heavy target tails. Very low and very high organic-content samples are especially vulnerable to regression-to-the-mean.

**Model choice.** CatBoost, LightGBM, and XGBoost are well suited for this dataset because they handle nonlinear interactions, missing values, mixed tabular signals, and high-cardinality categorical effects effectively. CatBoost is especially useful for native categorical handling, while LightGBM and XGBoost provide complementary gradient-boosted tree behavior.

**Leakage control.** The notebook uses a fixed target-stratified fold split for all models. Median imputation, ordinal encoding, target encoding, residual correction, calibration, and stacking are performed inside folds so validation targets never leak into training transformations.

**Tail-aware improvement.** The workflow explicitly evaluates RMSE and bias by target bin. It tests log-target models, tail-weighted training, residual correction, bin-specific calibration, and clipping strategies. A technique is only selected when OOF validation supports it.

**Final ensemble.** The final prediction is chosen from base models, weighted averages, stacking, and tail-aware calibrated variants. Selection balances global RMSE with stability on the very-low and very-high bins.

**Insights from EDA and errors.** The model investigates geography, source, biome, land cover, parent rock, chemistry, spectral PCs, band availability, and missingness as associated signals. Error analysis identifies where the model remains uncertain, especially rare source/biome regimes or incomplete spectral/chemical records.

**Limitations and next steps.** The notebook avoids external data, so it cannot use climate, elevation, soil maps, or satellite products. Future improvements could include better rare-tail sampling, standardized lab protocols, more complete spectral coverage, and source-aware leaderboard diagnostics.

## 12. Data Insights and Business Interpretation

The following interpretations should be read as associations, not causal claims.

- **Top associated factors.** Feature importance and EDA typically highlight source/geography fields, biome, land cover, parent rock, sampling depth, chemistry measurements, spectral PCs, and missingness/spectral availability features.
- **Biome, land cover, parent rock, source, and geo zones.** These categorical variables may capture ecological regime, sampling campaign, laboratory protocol, vegetation, geology, and location-specific soil formation processes. The model treats them as high-signal grouping variables while guarding against unseen categories.
- **Spectral PCs.** Spectral principal components summarize high-dimensional reflectance patterns. Their direct correlations may be modest individually, but tree models can combine them with chemistry, source, depth, and biome features to identify organic-content regimes.
- **Missing band B.** Missing spectral band B is modeled as information, not only as absent data. If band B availability differs by source or sampling protocol, the missingness itself can be predictive.
- **Soil chemistry and physical properties.** Particle fractions, acidity, cation measurements, and exchange capacity are likely associated with soil composition and organic matter retention. Ratio and interaction features help capture nonlinear relationships.
- **Why extremes are hard.** Very low and very high organic-content samples may be rare, source-specific, or tied to unusual biomes, land cover, depth, or chemistry patterns. Standard ensembles often shrink these cases toward the mean, which reduces average error but hurts tails.
- **Future data collection recommendations.** Reduce missing chemistry fields, standardize sampling depth notation, increase complete spectral coverage, improve geo metadata consistency, and collect more samples from rare high/low organic-content regimes.

## 13. Final Submission

The final predictions are aligned with `sample_submission`, checked for missing values, clipped at zero if the target is non-negative, and saved as `submission.csv`. Additional artifacts are also saved for diagnostics.

In [ ]:
# ============================================================
# 13. FINAL SUBMISSION + ORGANIZED SECTION DOWNLOADS
# ============================================================
# This final cell:
# 1. Builds submission.csv
# 2. Saves important CSV artifacts
# 3. Regenerates key plots/tables into section-based folders
# 4. Creates one ZIP per section
# 5. Creates one master ZIP containing all outputs
#
# Folder structure:
# competition_outputs/
# ├── 03_eda/
# ├── 04_feature_engineering/
# ├── 05_validation/
# ├── 06_supervised_learning/
# ├── 07_unsupervised_and_tuning/
# ├── 08_tail_aware_modeling/
# ├── 09_ensemble/
# ├── 10_visualization/
# └── 13_submission/
#
# ZIP files:
# download_zips/
# ├── 03_eda.zip
# ├── 04_feature_engineering.zip
# ├── ...
# └── all_competition_outputs.zip

import os
import gc
import json
import shutil
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")

try:
    from IPython.display import FileLink
    FILELINK_AVAILABLE = True
except Exception:
    FILELINK_AVAILABLE = False


# ============================================================
# 13.1 Output folder setup
# ============================================================

OUTPUT_ROOT = Path("competition_outputs")
ZIP_ROOT = Path("download_zips")

SECTION_DIRS = {
    "03_eda": OUTPUT_ROOT / "03_eda",
    "04_feature_engineering": OUTPUT_ROOT / "04_feature_engineering",
    "05_validation": OUTPUT_ROOT / "05_validation",
    "06_supervised_learning": OUTPUT_ROOT / "06_supervised_learning",
    "07_unsupervised_and_tuning": OUTPUT_ROOT / "07_unsupervised_and_tuning",
    "08_tail_aware_modeling": OUTPUT_ROOT / "08_tail_aware_modeling",
    "09_ensemble": OUTPUT_ROOT / "09_ensemble",
    "10_visualization": OUTPUT_ROOT / "10_visualization",
    "13_submission": OUTPUT_ROOT / "13_submission",
}

# Clean previous export folders to avoid mixing old and new files.
for folder in [OUTPUT_ROOT, ZIP_ROOT]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

for folder in SECTION_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)


def safe_filename(name):
    """
    Convert text into a filesystem-safe filename.
    """
    name = str(name)
    for ch in ['/', '\\', ':', '*', '?', '"', '<', '>', '|', '\n', '\t']:
        name = name.replace(ch, "_")
    name = name.replace(" ", "_")
    return name[:180]


def save_df(df, section_key, filename, index=False):
    """
    Save a dataframe into a specific section folder.
    Returns saved file path or None.
    """
    if df is None:
        return None

    if not isinstance(df, pd.DataFrame):
        try:
            df = pd.DataFrame(df)
        except Exception:
            return None

    path = SECTION_DIRS[section_key] / filename
    df.to_csv(path, index=index)
    return path


def save_series(series, section_key, filename, name=None):
    """
    Save a pandas Series into a specific section folder.
    """
    if series is None:
        return None

    if not isinstance(series, pd.Series):
        try:
            series = pd.Series(series)
        except Exception:
            return None

    if name is not None:
        series = series.rename(name)

    df = series.to_frame()
    path = SECTION_DIRS[section_key] / filename
    df.to_csv(path)
    return path


def save_json(obj, section_key, filename):
    """
    Save dictionary/list/JSON-like object into a section folder.
    """
    path = SECTION_DIRS[section_key] / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    return path


def save_text(text, section_key, filename):
    """
    Save plain text into a section folder.
    """
    path = SECTION_DIRS[section_key] / filename
    with open(path, "w", encoding="utf-8") as f:
        f.write(str(text))
    return path


def save_plot(fig, section_key, filename, dpi=160):
    """
    Save matplotlib figure into a section folder.
    """
    path = SECTION_DIRS[section_key] / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path


def rmse_np(y_true, y_pred):
    """
    Lightweight RMSE function.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def get_existing_var(name, default=None):
    """
    Safely get a variable from notebook globals.
    """
    return globals().get(name, default)


def detect_column(df, possible_names):
    """
    Detect first available column from a list of possible names.
    """
    if df is None or not isinstance(df, pd.DataFrame):
        return None

    lower_map = {c.lower(): c for c in df.columns}
    for name in possible_names:
        if name in df.columns:
            return name
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    return None


# ============================================================
# 13.2 Build final submission
# ============================================================

submission = sample_submission[[ID_COL]].copy()

pred_df = pd.DataFrame({
    ID_COL: test[ID_COL].values,
    TARGET: np.asarray(final_test_pred)
})

submission = submission.merge(pred_df, on=ID_COL, how="left")

assert submission[TARGET].notna().all(), "Submission contains NaN predictions."

# If target is non-negative in train, enforce non-negative predictions.
if np.nanmin(y) >= 0:
    submission[TARGET] = submission[TARGET].clip(lower=0)

# Save Kaggle-required submission at root.
submission.to_csv("submission.csv", index=False)

# Save another copy inside organized folder.
save_df(submission, "13_submission", "submission.csv", index=False)


# ============================================================
# 13.3 Save OOF predictions
# ============================================================

if "folds" in globals():
    folds_export = np.asarray(folds)
elif "fold" in train.columns:
    folds_export = train["fold"].values
else:
    folds_export = np.full(len(train), -1)

oof_out = pd.DataFrame({
    ID_COL: train[ID_COL].values,
    TARGET: np.asarray(y),
    "fold": folds_export,
    "final_oof_prediction": np.asarray(final_oof),
})

if "candidate_oof" in globals() and isinstance(candidate_oof, dict):
    for name, pred in candidate_oof.items():
        clean_name = safe_filename(name)
        if len(pred) == len(oof_out):
            oof_out[f"oof_{clean_name}"] = np.asarray(pred)

save_df(oof_out, "13_submission", "oof_predictions.csv", index=False)
save_df(oof_out, "09_ensemble", "oof_predictions_for_ensemble.csv", index=False)


# ============================================================
# 13.4 Save model scores and reports
# ============================================================

if "candidate_scores" in globals() and isinstance(candidate_scores, pd.DataFrame):
    save_df(candidate_scores, "06_supervised_learning", "model_cv_scores.csv", index=False)
    save_df(candidate_scores, "09_ensemble", "candidate_ensemble_scores.csv", index=False)
else:
    candidate_scores = pd.DataFrame()

if "final_report" in globals() and isinstance(final_report, pd.DataFrame):
    save_df(final_report, "08_tail_aware_modeling", "final_error_report.csv", index=False)
else:
    final_report = pd.DataFrame()

if "error_df" in globals() and isinstance(error_df, pd.DataFrame):
    save_df(error_df, "08_tail_aware_modeling", "final_error_analysis.csv", index=False)
else:
    # Build a basic error dataframe if it does not already exist.
    error_df = pd.DataFrame({
        ID_COL: train[ID_COL].values,
        "actual": np.asarray(y),
        "prediction": np.asarray(final_oof),
    })
    error_df["residual"] = error_df["prediction"] - error_df["actual"]
    error_df["abs_error"] = error_df["residual"].abs()
    error_df["squared_error"] = error_df["residual"] ** 2

    if "source_id" in train.columns:
        error_df["source_id"] = train["source_id"].astype(str).values
    if "biome" in train.columns:
        error_df["biome"] = train["biome"].astype(str).values
    if "has_band_B_spectrum" in train.columns:
        error_df["has_band_B_spectrum"] = train["has_band_B_spectrum"].astype(str).values

    save_df(error_df, "08_tail_aware_modeling", "final_error_analysis.csv", index=False)


# ============================================================
# 13.5 Save feature importance artifacts
# ============================================================

importance_files = []

if "importance_summary" in globals() and isinstance(importance_summary, pd.DataFrame) and not importance_summary.empty:
    save_df(importance_summary, "06_supervised_learning", "feature_importance_lgb_xgb.csv", index=False)
    save_df(importance_summary, "10_visualization", "feature_importance_lgb_xgb.csv", index=False)
    importance_files.append("feature_importance_lgb_xgb.csv")

if "cb_imp_summary" in globals() and isinstance(cb_imp_summary, pd.DataFrame) and not cb_imp_summary.empty:
    save_df(cb_imp_summary, "06_supervised_learning", "feature_importance_catboost.csv", index=False)
    save_df(cb_imp_summary, "10_visualization", "feature_importance_catboost.csv", index=False)
    importance_files.append("feature_importance_catboost.csv")


# ============================================================
# 13.6 Save EDA tables
# ============================================================

# Missingness comparison table.
all_cols = sorted(set(train.columns).union(set(test.columns)))

missing_rows = []
for col in all_cols:
    train_missing_count = train[col].isna().sum() if col in train.columns else np.nan
    test_missing_count = test[col].isna().sum() if col in test.columns else np.nan

    train_missing_pct = train[col].isna().mean() * 100 if col in train.columns else np.nan
    test_missing_pct = test[col].isna().mean() * 100 if col in test.columns else np.nan

    missing_rows.append({
        "column": col,
        "train_missing_count": train_missing_count,
        "train_missing_pct": train_missing_pct,
        "test_missing_count": test_missing_count,
        "test_missing_pct": test_missing_pct,
        "missing_pct_diff_test_minus_train": (
            test_missing_pct - train_missing_pct
            if pd.notna(test_missing_pct) and pd.notna(train_missing_pct)
            else np.nan
        ),
    })

missing_report = pd.DataFrame(missing_rows).sort_values(
    ["train_missing_pct", "test_missing_pct"],
    ascending=False
)

save_df(missing_report, "03_eda", "missingness_train_test_comparison.csv", index=False)

# Data type summary.
dtype_report = pd.DataFrame({
    "column": train.columns,
    "dtype": [str(train[c].dtype) for c in train.columns],
    "n_unique_train": [train[c].nunique(dropna=False) for c in train.columns],
    "n_missing_train": [train[c].isna().sum() for c in train.columns],
    "missing_pct_train": [train[c].isna().mean() * 100 for c in train.columns],
})

save_df(dtype_report, "03_eda", "train_dtype_cardinality_summary.csv", index=False)

# Target quantiles.
target_quantiles = pd.Series(y).quantile(
    [0, .001, .01, .05, .10, .25, .50, .75, .90, .95, .99, .999, 1.0]
).rename("target_quantile")

save_series(target_quantiles, "03_eda", "target_quantiles.csv")

# Target summary.
target_summary = pd.Series(y).describe(
    percentiles=[.001, .01, .05, .10, .25, .50, .75, .90, .95, .99, .999]
).rename("target_summary")

save_series(target_summary, "03_eda", "target_summary_statistics.csv")

# Categorical level comparison.
if "categorical_cols" in globals():
    cat_cols_export = categorical_cols
else:
    cat_cols_export = [
        c for c in train.columns
        if train[c].dtype == "object" or str(train[c].dtype) == "category"
    ]
    cat_cols_export = [c for c in cat_cols_export if c not in [TARGET, ID_COL]]

cat_level_rows = []
for col in cat_cols_export:
    if col in train.columns and col in test.columns:
        train_levels = set(train[col].astype(str).fillna("Missing").unique())
        test_levels = set(test[col].astype(str).fillna("Missing").unique())

        cat_level_rows.append({
            "column": col,
            "train_n_levels": len(train_levels),
            "test_n_levels": len(test_levels),
            "unseen_in_test_count": len(test_levels - train_levels),
            "missing_from_test_count": len(train_levels - test_levels),
            "unseen_in_test_examples": ", ".join(list(test_levels - train_levels)[:20]),
            "missing_from_test_examples": ", ".join(list(train_levels - test_levels)[:20]),
        })

cat_level_report = pd.DataFrame(cat_level_rows)
save_df(cat_level_report, "03_eda", "categorical_level_train_test_comparison.csv", index=False)


# ============================================================
# 13.7 Save feature engineering metadata
# ============================================================

feature_group_names = [
    "categorical_cols",
    "numeric_cols",
    "geo_cols",
    "spectral_A_cols",
    "spectral_B_cols",
    "soil_numeric_cols",
    "binary_flag_cols",
    "model_feature_cols",
    "target_encode_cols",
    "frequency_encode_cols",
]

feature_groups_export = {}
for var_name in feature_group_names:
    value = get_existing_var(var_name, None)
    if value is not None:
        try:
            feature_groups_export[var_name] = list(value)
        except Exception:
            feature_groups_export[var_name] = str(value)

if feature_groups_export:
    save_json(feature_groups_export, "04_feature_engineering", "feature_groups.json")

if "X_train_model" in globals() and isinstance(X_train_model, pd.DataFrame):
    feature_metadata = pd.DataFrame({
        "feature": X_train_model.columns,
        "dtype": [str(X_train_model[c].dtype) for c in X_train_model.columns],
        "missing_count": [X_train_model[c].isna().sum() for c in X_train_model.columns],
        "missing_pct": [X_train_model[c].isna().mean() * 100 for c in X_train_model.columns],
        "n_unique": [X_train_model[c].nunique(dropna=False) for c in X_train_model.columns],
    })
    save_df(feature_metadata, "04_feature_engineering", "engineered_feature_metadata.csv", index=False)


# ============================================================
# 13.8 Save validation diagnostics
# ============================================================

fold_report = pd.DataFrame({
    ID_COL: train[ID_COL].values,
    TARGET: np.asarray(y),
    "fold": folds_export,
})

save_df(fold_report, "05_validation", "fold_assignment.csv", index=False)

if len(np.unique(folds_export)) > 1:
    fold_target_summary = (
        fold_report
        .groupby("fold")[TARGET]
        .describe(percentiles=[.05, .10, .50, .90, .95])
        .reset_index()
    )
    save_df(fold_target_summary, "05_validation", "fold_target_distribution_summary.csv", index=False)


# ============================================================
# 13.9 Save ensemble metadata
# ============================================================

final_cv_rmse = rmse_np(np.asarray(y), np.asarray(final_oof))

final_summary = {
    "final_selected_approach": str(get_existing_var("FINAL_NAME", "final_model")),
    "final_cv_rmse": final_cv_rmse,
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "target": TARGET,
    "id_col": ID_COL,
    "submission_file": "submission.csv",
}

save_json(final_summary, "13_submission", "final_summary.json")
save_json(final_summary, "09_ensemble", "final_summary.json")

if "ensemble_details" in globals() and ensemble_details:
    try:
        weights = ensemble_details[0].get("weights", {})
        weights_df = pd.Series(weights).sort_values(ascending=False).to_frame("weight")
        weights_df.index.name = "model"
        save_df(weights_df.reset_index(), "09_ensemble", "final_ensemble_weights.csv", index=False)
    except Exception:
        pass


# ============================================================
# 13.10 Generate and save important plots
# ============================================================

# 1. Missingness bar chart.
try:
    plot_df = missing_report.copy()
    plot_df["max_missing_pct"] = plot_df[["train_missing_pct", "test_missing_pct"]].max(axis=1)
    plot_df = plot_df.sort_values("max_missing_pct", ascending=False).head(30)

    fig, ax = plt.subplots(figsize=(12, 7))
    x = np.arange(len(plot_df))
    width = 0.40

    ax.bar(x - width / 2, plot_df["train_missing_pct"], width, label="train")
    ax.bar(x + width / 2, plot_df["test_missing_pct"], width, label="test")

    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["column"], rotation=75, ha="right")
    ax.set_ylabel("Missing percentage")
    ax.set_title("Top Missing Features: Train vs Test")
    ax.legend()

    save_plot(fig, "03_eda", "missingness_train_test_top30.png")
except Exception as e:
    save_text(f"Could not create missingness plot: {e}", "03_eda", "missingness_plot_error.txt")


# 2. Target distribution.
try:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(y, bins=50)
    ax.set_title("Target Distribution")
    ax.set_xlabel(TARGET)
    ax.set_ylabel("Count")
    save_plot(fig, "03_eda", "target_distribution.png")
except Exception as e:
    save_text(f"Could not create target distribution plot: {e}", "03_eda", "target_distribution_error.txt")


# 3. log1p target distribution.
try:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(np.log1p(y), bins=50)
    ax.set_title("log1p(Target) Distribution")
    ax.set_xlabel(f"log1p({TARGET})")
    ax.set_ylabel("Count")
    save_plot(fig, "03_eda", "log1p_target_distribution.png")
except Exception as e:
    save_text(f"Could not create log1p target distribution plot: {e}", "03_eda", "log1p_target_distribution_error.txt")


# 4. Target boxplot.
try:
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.boxplot(y, vert=False)
    ax.set_title("Target Boxplot")
    ax.set_xlabel(TARGET)
    save_plot(fig, "03_eda", "target_boxplot.png")
except Exception as e:
    save_text(f"Could not create target boxplot: {e}", "03_eda", "target_boxplot_error.txt")


# 5. Actual vs predicted.
try:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y, final_oof, s=10, alpha=0.45)

    min_v = min(np.nanmin(y), np.nanmin(final_oof))
    max_v = max(np.nanmax(y), np.nanmax(final_oof))
    ax.plot([min_v, max_v], [min_v, max_v], linestyle="--")

    ax.set_title("Actual vs OOF Prediction")
    ax.set_xlabel("Actual")
    ax.set_ylabel("OOF Prediction")

    save_plot(fig, "09_ensemble", "actual_vs_oof_prediction.png")
    save_plot(fig, "10_visualization", "actual_vs_oof_prediction.png")
except Exception as e:
    save_text(f"Could not create actual vs prediction plot: {e}", "09_ensemble", "actual_vs_prediction_error.txt")


# 6. Residual distribution.
try:
    residuals = np.asarray(final_oof) - np.asarray(y)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(residuals, bins=60)
    ax.axvline(0, linestyle="--")
    ax.set_title("Residual Distribution")
    ax.set_xlabel("Prediction - Actual")
    ax.set_ylabel("Count")

    save_plot(fig, "08_tail_aware_modeling", "residual_distribution.png")
    save_plot(fig, "10_visualization", "residual_distribution.png")
except Exception as e:
    save_text(f"Could not create residual distribution plot: {e}", "08_tail_aware_modeling", "residual_distribution_error.txt")


# 7. Residual vs actual.
try:
    residuals = np.asarray(final_oof) - np.asarray(y)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(y, residuals, s=10, alpha=0.45)
    ax.axhline(0, linestyle="--")
    ax.set_title("Residual vs Actual Target")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Prediction - Actual")

    save_plot(fig, "08_tail_aware_modeling", "residual_vs_actual.png")
    save_plot(fig, "10_visualization", "residual_vs_actual.png")
except Exception as e:
    save_text(f"Could not create residual vs actual plot: {e}", "08_tail_aware_modeling", "residual_vs_actual_error.txt")


# 8. RMSE by target bin.
try:
    if not final_report.empty and "segment" in final_report.columns:
        bin_report = final_report[final_report["segment"].astype(str) == "target_bin"].copy()
    else:
        bin_report = pd.DataFrame()

    if bin_report.empty:
        temp_error = error_df.copy()

        if "actual" not in temp_error.columns:
            temp_error["actual"] = np.asarray(y)
        if "prediction" not in temp_error.columns:
            temp_error["prediction"] = np.asarray(final_oof)

        q05, q10, q90, q95 = pd.Series(y).quantile([.05, .10, .90, .95])

        def assign_tail_bin(v):
            if v <= q05:
                return "very_low"
            if v <= q10:
                return "low"
            if v >= q95:
                return "very_high"
            if v >= q90:
                return "high"
            return "mid"

        temp_error["target_bin"] = temp_error["actual"].apply(assign_tail_bin)
        temp_error["squared_error"] = (temp_error["prediction"] - temp_error["actual"]) ** 2

        bin_report = (
            temp_error
            .groupby("target_bin")
            .agg(
                n=("actual", "size"),
                rmse=("squared_error", lambda s: float(np.sqrt(np.mean(s)))),
                bias=("prediction", lambda p: float(np.mean(p - temp_error.loc[p.index, "actual"]))),
            )
            .reset_index()
        )

        save_df(bin_report, "08_tail_aware_modeling", "final_rmse_by_target_bin_recomputed.csv", index=False)

    rmse_col = detect_column(bin_report, ["rmse", "RMSE"])
    bin_col = detect_column(bin_report, ["target_bin", "bin", "group", "value", "segment_value"])

    if rmse_col is not None and bin_col is not None:
        fig, ax = plt.subplots(figsize=(9, 5))
        plot_df = bin_report.copy()
        ax.bar(plot_df[bin_col].astype(str), plot_df[rmse_col])
        ax.set_title("Final RMSE by Target Bin")
        ax.set_xlabel("Target bin")
        ax.set_ylabel("RMSE")
        ax.tick_params(axis="x", rotation=30)

        save_plot(fig, "08_tail_aware_modeling", "final_rmse_by_target_bin.png")
        save_plot(fig, "10_visualization", "final_rmse_by_target_bin.png")

except Exception as e:
    save_text(f"Could not create RMSE by target bin plot: {e}", "08_tail_aware_modeling", "rmse_by_target_bin_error.txt")


# 9. Model comparison chart.
try:
    if not candidate_scores.empty:
        score_df = candidate_scores.copy()

        model_col = detect_column(score_df, ["model", "name", "candidate", "approach"])
        score_col = detect_column(score_df, ["rmse", "cv_rmse", "oof_rmse", "score"])

        if model_col is not None and score_col is not None:
            plot_df = score_df[[model_col, score_col]].dropna().copy()
            plot_df = plot_df.sort_values(score_col, ascending=True).head(30)

            fig, ax = plt.subplots(figsize=(11, 6))
            ax.barh(plot_df[model_col].astype(str), plot_df[score_col])
            ax.invert_yaxis()
            ax.set_title("Model / Ensemble CV RMSE Comparison")
            ax.set_xlabel("RMSE")

            save_plot(fig, "09_ensemble", "model_cv_rmse_comparison.png")
            save_plot(fig, "10_visualization", "model_cv_rmse_comparison.png")

except Exception as e:
    save_text(f"Could not create model comparison plot: {e}", "09_ensemble", "model_comparison_error.txt")


# 10. Ensemble weights chart.
try:
    if "ensemble_details" in globals() and ensemble_details:
        weights = ensemble_details[0].get("weights", {})
        weights_series = pd.Series(weights).sort_values(ascending=True)

        if len(weights_series) > 0:
            fig, ax = plt.subplots(figsize=(9, max(4, len(weights_series) * 0.4)))
            ax.barh(weights_series.index.astype(str), weights_series.values)
            ax.set_title("Final / Best Weighted Ensemble Weights")
            ax.set_xlabel("Weight")

            save_plot(fig, "09_ensemble", "final_ensemble_weights.png")
            save_plot(fig, "10_visualization", "final_ensemble_weights.png")

except Exception as e:
    save_text(f"Could not create ensemble weights plot: {e}", "09_ensemble", "ensemble_weights_error.txt")


# 11. Prediction distribution: train OOF vs test final.
try:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(final_oof, bins=50, alpha=0.5, label="OOF predictions")
    ax.hist(submission[TARGET], bins=50, alpha=0.5, label="Test predictions")
    ax.set_title("Prediction Distribution: OOF vs Test")
    ax.set_xlabel(TARGET)
    ax.set_ylabel("Count")
    ax.legend()

    save_plot(fig, "13_submission", "prediction_distribution_oof_vs_test.png")
    save_plot(fig, "10_visualization", "prediction_distribution_oof_vs_test.png")
except Exception as e:
    save_text(f"Could not create prediction distribution plot: {e}", "13_submission", "prediction_distribution_error.txt")


# 12. Feature importance chart.
try:
    imp_df = None

    if "importance_summary" in globals() and isinstance(importance_summary, pd.DataFrame) and not importance_summary.empty:
        imp_df = importance_summary.copy()
    elif "cb_imp_summary" in globals() and isinstance(cb_imp_summary, pd.DataFrame) and not cb_imp_summary.empty:
        imp_df = cb_imp_summary.copy()

    if imp_df is not None:
        feature_col = detect_column(imp_df, ["feature", "Feature", "Feature Id", "feature_name"])
        importance_col = detect_column(imp_df, ["importance", "mean_importance", "Importance", "gain", "split"])

        if feature_col is not None and importance_col is not None:
            plot_df = imp_df[[feature_col, importance_col]].dropna().copy()
            plot_df = plot_df.sort_values(importance_col, ascending=False).head(30)
            plot_df = plot_df.sort_values(importance_col, ascending=True)

            fig, ax = plt.subplots(figsize=(10, 9))
            ax.barh(plot_df[feature_col].astype(str), plot_df[importance_col])
            ax.set_title("Top 30 Feature Importances")
            ax.set_xlabel("Importance")

            save_plot(fig, "06_supervised_learning", "top30_feature_importance.png")
            save_plot(fig, "10_visualization", "top30_feature_importance.png")

except Exception as e:
    save_text(f"Could not create feature importance plot: {e}", "06_supervised_learning", "feature_importance_plot_error.txt")


# 13. Source-level error plot.
try:
    if "source_id" in error_df.columns:
        temp = error_df.copy()

        actual_col = detect_column(temp, ["actual", TARGET])
        pred_col = detect_column(temp, ["prediction", "final_oof_prediction", "pred"])

        if actual_col is not None and pred_col is not None:
            temp["squared_error_tmp"] = (temp[pred_col] - temp[actual_col]) ** 2

            source_error = (
                temp
                .groupby("source_id")
                .agg(
                    n=(actual_col, "size"),
                    rmse=("squared_error_tmp", lambda s: float(np.sqrt(np.mean(s)))),
                    mean_actual=(actual_col, "mean"),
                    mean_prediction=(pred_col, "mean"),
                )
                .reset_index()
                .sort_values("rmse", ascending=False)
            )

            save_df(source_error, "08_tail_aware_modeling", "rmse_by_source_id.csv", index=False)

            plot_df = source_error[source_error["n"] >= 10].head(25)

            if not plot_df.empty:
                plot_df = plot_df.sort_values("rmse", ascending=True)

                fig, ax = plt.subplots(figsize=(10, 8))
                ax.barh(plot_df["source_id"].astype(str), plot_df["rmse"])
                ax.set_title("RMSE by source_id, min n=10")
                ax.set_xlabel("RMSE")

                save_plot(fig, "08_tail_aware_modeling", "rmse_by_source_id_top25.png")
                save_plot(fig, "10_visualization", "rmse_by_source_id_top25.png")

except Exception as e:
    save_text(f"Could not create source-level error plot: {e}", "08_tail_aware_modeling", "source_error_plot_error.txt")


# 14. Spectral availability error plot.
try:
    if "has_band_B_spectrum" in error_df.columns:
        temp = error_df.copy()

        actual_col = detect_column(temp, ["actual", TARGET])
        pred_col = detect_column(temp, ["prediction", "final_oof_prediction", "pred"])

        if actual_col is not None and pred_col is not None:
            temp["squared_error_tmp"] = (temp[pred_col] - temp[actual_col]) ** 2

            band_error = (
                temp
                .groupby("has_band_B_spectrum")
                .agg(
                    n=(actual_col, "size"),
                    rmse=("squared_error_tmp", lambda s: float(np.sqrt(np.mean(s)))),
                    mean_actual=(actual_col, "mean"),
                    mean_prediction=(pred_col, "mean"),
                )
                .reset_index()
                .sort_values("rmse", ascending=False)
            )

            save_df(band_error, "08_tail_aware_modeling", "rmse_by_has_band_B_spectrum.csv", index=False)

            fig, ax = plt.subplots(figsize=(7, 4))
            ax.bar(band_error["has_band_B_spectrum"].astype(str), band_error["rmse"])
            ax.set_title("RMSE by Band B Spectrum Availability")
            ax.set_xlabel("has_band_B_spectrum")
            ax.set_ylabel("RMSE")

            save_plot(fig, "08_tail_aware_modeling", "rmse_by_band_B_availability.png")
            save_plot(fig, "10_visualization", "rmse_by_band_B_availability.png")

except Exception as e:
    save_text(f"Could not create spectral availability error plot: {e}", "08_tail_aware_modeling", "spectral_availability_error_plot_error.txt")


# ============================================================
# 13.11 Save prediction summary statistics
# ============================================================

prediction_summary = submission[TARGET].describe(
    percentiles=[.001, .01, .05, .10, .25, .50, .75, .90, .95, .99, .999]
).rename("submission_prediction_summary")

save_series(prediction_summary, "13_submission", "submission_prediction_summary.csv")

submission_head = submission.head(20)
save_df(submission_head, "13_submission", "submission_head.csv", index=False)


# ============================================================
# 13.12 Create ZIP files per section
# ============================================================

def count_files(folder):
    """
    Count files inside folder recursively.
    """
    folder = Path(folder)
    return sum(1 for p in folder.rglob("*") if p.is_file())


def get_folder_size_mb(folder):
    """
    Get total folder size in MB.
    """
    folder = Path(folder)
    total = 0
    for p in folder.rglob("*"):
        if p.is_file():
            total += p.stat().st_size
    return total / (1024 ** 2)


def zip_folder(folder_path, zip_path):
    """
    Zip an entire folder.
    """
    folder_path = Path(folder_path)
    zip_path = Path(zip_path)

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for file_path in folder_path.rglob("*"):
            if file_path.is_file():
                arcname = file_path.relative_to(folder_path.parent)
                zf.write(file_path, arcname)

    return zip_path


zip_manifest_rows = []

for section_key, folder_path in SECTION_DIRS.items():
    n_files = count_files(folder_path)

    if n_files == 0:
        continue

    zip_path = ZIP_ROOT / f"{section_key}.zip"
    zip_folder(folder_path, zip_path)

    zip_manifest_rows.append({
        "section": section_key,
        "folder": str(folder_path),
        "zip_file": str(zip_path),
        "num_files": n_files,
        "folder_size_mb": get_folder_size_mb(folder_path),
        "zip_size_mb": zip_path.stat().st_size / (1024 ** 2),
    })


# Master ZIP containing every section output.
master_zip_path = ZIP_ROOT / "all_competition_outputs.zip"
zip_folder(OUTPUT_ROOT, master_zip_path)

zip_manifest_rows.append({
    "section": "all_sections",
    "folder": str(OUTPUT_ROOT),
    "zip_file": str(master_zip_path),
    "num_files": count_files(OUTPUT_ROOT),
    "folder_size_mb": get_folder_size_mb(OUTPUT_ROOT),
    "zip_size_mb": master_zip_path.stat().st_size / (1024 ** 2),
})

zip_manifest = pd.DataFrame(zip_manifest_rows)
zip_manifest.to_csv(ZIP_ROOT / "zip_manifest.csv", index=False)
save_df(zip_manifest, "13_submission", "zip_manifest.csv", index=False)


# ============================================================
# 13.13 Print final summary
# ============================================================

print("=" * 80)
print("FINAL SELECTED APPROACH")
print("=" * 80)
print("Final selected approach:", get_existing_var("FINAL_NAME", "final_model"))
print("Final CV RMSE:", final_cv_rmse)

print("\n" + "=" * 80)
print("FINAL RMSE BY TARGET BIN")
print("=" * 80)

if not final_report.empty and "segment" in final_report.columns:
    display(final_report[final_report["segment"].astype(str) == "target_bin"])
else:
    print("final_report with target_bin segment is not available. Recomputed bin report may be saved if possible.")

print("\n" + "=" * 80)
print("FINAL / AVAILABLE ENSEMBLE WEIGHTS")
print("=" * 80)

if "ensemble_details" in globals() and ensemble_details:
    try:
        display(pd.Series(ensemble_details[0]["weights"]).sort_values(ascending=False).to_frame("weight"))
    except Exception:
        print("Ensemble details found, but weights could not be displayed.")
else:
    print("No optimized weighted ensemble details available.")

print("\n" + "=" * 80)
print("PREDICTION SUMMARY STATISTICS")
print("=" * 80)
display(submission[TARGET].describe(
    percentiles=[.001, .01, .05, .10, .25, .50, .75, .90, .95, .99, .999]
).to_frame("submission_pred"))

print("\n" + "=" * 80)
print("SUBMISSION HEAD")
print("=" * 80)
display(submission.head())

print("\n" + "=" * 80)
print("SAVED ROOT FILE")
print("=" * 80)
print("- submission.csv")

print("\n" + "=" * 80)
print("SECTION ZIP FILES")
print("=" * 80)
display(zip_manifest)

print("\nDownload files created in:", str(ZIP_ROOT.resolve()))
print("Master ZIP:", str(master_zip_path.resolve()))


# ============================================================
# 13.14 Display clickable download links when supported
# ============================================================

if FILELINK_AVAILABLE:
    print("\nClickable download links:")

    # Kaggle/Jupyter usually supports FileLink for local notebook files.
    for _, row in zip_manifest.iterrows():
        zip_file = row["zip_file"]
        print(f"\n{row['section']}:")
        display(FileLink(zip_file))

    print("\nMain Kaggle submission file:")
    display(FileLink("submission.csv"))

else:
    print("\nFileLink is not available in this environment.")
    print("You can manually download files from:")
    print("- submission.csv")
    print(f"- {ZIP_ROOT}/")


gc.collect()

## 14. Notebook Quality Checklist

- End-to-end runnable.
- Clean markdown sections and modular reusable functions.
- No external data.
- No AutoML or automated feature engineering libraries.
- Fixed fold assignment shared by all models.
- Fold-safe target encoding, imputation, ordinal encoding, stacking, residual correction, and calibration.
- Robust missing-value handling.
- Tail-bin RMSE and bias diagnostics.
- Final selection based on validation, not wishful thinking.
- Produces `submission.csv` and diagnostic artifacts.